# AutoGrow4 on SDSC Expanse

This notebook provides a complete environment for running **AutoGrow 4.0** on the SDSC Expanse supercomputer.  
It includes:
1. **Installation** of AutoGrow4 and all dependencies
2. **NGL Viewer** for receptor visualization and binding-box definition
3. **Interactive GUI** (ipywidgets) for configuring all AutoGrow4 parameters
4. **Slurm job builder** that generates and submits SBATCH scripts
5. **Job monitoring** to track progress and retrieve results

---

## 1. Installation & Setup

Run the cells below **once** to install AutoGrow4 and all required dependencies.  
These are designed for the SDSC Expanse environment (Linux, module system, conda).

In [9]:
import os
import sys

# === Configuration ===
# Set your project directory (where everything will be installed)
PROJECT_DIR = os.path.expanduser("~/final_project")
AUTOGROW_DIR = os.path.join(PROJECT_DIR, "autogrow4")
MGLTOOLS_DIR = os.path.join(PROJECT_DIR, "mgltools_x86_64Linux2_1.5.7")
# Use Lustre scratch for better I/O performance on Expanse
OUTPUT_DIR = os.path.expanduser("~/lustre_scratch/autogrow_output")
# Lustre scratch symlink: ~/lustre_scratch -> /expanse/lustre/scratch/$USER/temp_project
# If not set up yet, create it: ln -s /expanse/lustre/scratch/$USER/temp_project ~/lustre_scratch
RECEPTOR_DIR = os.path.join(PROJECT_DIR, "receptors")

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RECEPTOR_DIR, exist_ok=True)

# Default receptor: GID1 (Gibberellin Insensitive Dwarf 1)
DEFAULT_RECEPTOR = os.path.join(RECEPTOR_DIR, "2ZSH.pdb")

print(f"Project directory: {PROJECT_DIR}")
print(f"AutoGrow4 directory: {AUTOGROW_DIR}")
print(f"Receptor directory: {RECEPTOR_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

Project directory: /home/mschnur/final_project
AutoGrow4 directory: /home/mschnur/final_project/autogrow4
Receptor directory: /home/mschnur/final_project/receptors
Output directory: /home/mschnur/lustre_scratch/autogrow_output


### 1.1 Install Python Dependencies (conda + pip)

In [ ]:
%%bash
# Install core Python dependencies via conda
conda install -y -c conda-forge rdkit numpy scipy matplotlib func_timeout 2>&1 | tail -5
echo "--- Core Python packages installed ---"

# Install mpi4py (required for supercomputer MPI runs)
conda install -y -c conda-forge mpi4py openmpi 2>&1 | tail -5
echo "--- mpi4py installed ---"

# Install nglview for molecular visualization
conda install -y -c conda-forge nglview 2>&1 | tail -5
echo "--- nglview installed ---"

# Install ipywidgets for the GUI
conda install -y -c conda-forge ipywidgets 2>&1 | tail -5
echo "--- ipywidgets installed ---"

# Install scoria for binding box calculations
pip install scoria 2>&1 | tail -3
echo "--- scoria installed ---"

echo "\n=== All Python dependencies installed ==="

### 1.2 Install OpenBabel

In [10]:
%%bash
# Install OpenBabel via conda (preferred on Expanse — no sudo needed)
conda install -y -c conda-forge openbabel 2>&1 | tail -5

# Verify installation
echo "OpenBabel path: $(which obabel)"
obabel -V 2>&1 | head -1




# All requested packages already installed.

OpenBabel path: /home/mschnur/miniconda3/envs/visualization/bin/obabel
Open Babel 2.4.1 -- Jan 25 2017 -- 17:44:52


### 1.3 Install MGLTools (Manual — NOT via pip/conda)

**Important:** MGLTools must be installed from the official tarball, NOT via pip or conda.  
This downloads the command-line version from Scripps.

In [2]:
%%bash
cd ~/final_project

if [ ! -d "mgltools_x86_64Linux2_1.5.7" ]; then
    echo "Downloading MGLTools 1.5.6..."
    wget -q http://mgltools.scripps.edu/downloads/downloads/tars/releases/REL1.5.6/mgltools_x86_64Linux2_1.5.7.tar.gz
    
    echo "Extracting..."
    tar -xzf mgltools_x86_64Linux2_1.5.7.tar.gz
    
    echo "Installing MGLTools..."
    cd mgltools_x86_64Linux2_1.5.7
    
    # Check if MGLToolsPckgs needs to be extracted
    if [ ! -d "MGLToolsPckgs" ] && [ -f "MGLToolsPckgs.tar.gz" ]; then
        tar -xzf MGLToolsPckgs.tar.gz
    fi
    
    # Run install (non-interactive)
    echo "y" | bash install.sh 2>&1 | tail -5
    
    cd ..
    rm -f mgltools_x86_64Linux2_1.5.7.tar.gz
    echo "MGLTools installed successfully."
else
    echo "MGLTools already installed."
fi

# Verify key files exist
ls ~/final_project/mgltools_x86_64Linux2_1.5.7/bin/pythonsh 2>/dev/null && echo "✓ pythonsh found" || echo "✗ pythonsh NOT found"
ls ~/final_project/mgltools_x86_64Linux2_1.5.7/MGLToolsPckgs/AutoDockTools/Utilities24/prepare_ligand4.py 2>/dev/null && echo "✓ prepare_ligand4.py found" || echo "✗ prepare_ligand4.py NOT found"

MGLTools already installed.
/home/mschnur/final_project/mgltools_x86_64Linux2_1.5.7/bin/pythonsh
✓ pythonsh found
/home/mschnur/final_project/mgltools_x86_64Linux2_1.5.7/MGLToolsPckgs/AutoDockTools/Utilities24/prepare_ligand4.py
✓ prepare_ligand4.py found


### 1.4 Clone AutoGrow4

In [3]:
%%bash
cd ~/final_project

if [ ! -d "autogrow4" ]; then
    echo "Cloning AutoGrow4..."
    git clone https://github.com/durrantlab/autogrow4.git 2>&1 | tail -3
    echo "AutoGrow4 cloned successfully."
else
    echo "AutoGrow4 already present."
fi

# Verify
ls ~/final_project/autogrow4/run_autogrow.py && echo "✓ run_autogrow.py found"

AutoGrow4 already present.
/home/mschnur/final_project/autogrow4/run_autogrow.py
✓ run_autogrow.py found


### 1.4a Apply Compatibility Patches

AutoGrow4 requires patches for compatibility with **RDKit 2025+** and the `CompoundInfo` dataclass refactoring.

In [11]:
%%bash
cd ~/final_project/autogrow4

echo "=== Applying AutoGrow4 compatibility patches ==="

# Patch 1: MyMol.py - RDKit 2025+ chirality parameter
MYMOL=autogrow/operators/convert_files/gypsum_dl/gypsum_dl/MyMol.py
python3 << 'PATCH1'
import re
path = "autogrow/operators/convert_files/gypsum_dl/gypsum_dl/MyMol.py"
content = open(path).read()
clean = "params.enforceChirality = True  # RDKit 2025+"
old_only = "params.enforcechiral = True"
if clean in content and content.count("try:") - content.count("except") == 0:
    print("  [SKIP] MyMol.py already cleanly patched")
else:
    pattern = r"(            # The default, but just a sanity check\.\n)(.*?)(            # Set a max number of times)"
    m = re.search(pattern, content, re.DOTALL)
    if m:
        repl = (m.group(1) +
            "            try:\n"
            "                params.enforceChirality = True  # RDKit 2025+\n"
            "            except AttributeError:\n"
            "                params.enforcechiral = True  # Older RDKit\n\n"
            + m.group(3))
        content = content[:m.start()] + repl + content[m.end():]
        open(path, "w").write(content)
        print("  [OK] MyMol.py: clean chirality patch")
    elif old_only in content:
        content = content.replace(
            "            params.enforcechiral = True",
            "            try:\n"
            "                params.enforceChirality = True  # RDKit 2025+\n"
            "            except AttributeError:\n"
            "                params.enforcechiral = True  # Older RDKit")
        open(path, "w").write(content)
        print("  [OK] MyMol.py: basic chirality patch")
    else:
        print("  [SKIP] MyMol.py: no enforcechiral found")
PATCH1

# Patch 2: types.py - CompoundInfo backward compat + score fix + append
TYPES=autogrow/types.py
python3 << 'PATCH2'
path = "autogrow/types.py"
content = open(path).read()
changed = False

if "__getitem__" not in content:
    new_methods = """
    def __getitem__(self, index):
        lst = self.to_list()
        return lst[index]

    def __len__(self):
        return len(self.to_list())

    def __bool__(self):
        return True

    def __iter__(self):
        return iter(self.to_list())

    def append(self, value):
        self.diversity_score = float(value)

"""
    content = content.replace(
        "    @property\n    def score",
        new_methods + "    @property\n    def score"
    )
    changed = True
    print("  [OK] types.py: added backward compat methods")

if "def append" not in content:
    content = content.replace(
        "    @property\n    def score",
        "    def append(self, value):\n"
        "        self.diversity_score = float(value)\n\n"
        "    @property\n    def score"
    )
    changed = True
    print("  [OK] types.py: added append method")

if "if self.docking_score:" in content:
    content = content.replace("if self.docking_score:", "if self.docking_score is not None:")
    content = content.replace("if self.diversity_score:", "if self.diversity_score is not None:")
    content = content.replace("if self.fitness_score:", "if self.fitness_score is not None:")
    changed = True
    print("  [OK] types.py: fixed score property truthiness")

if changed:
    open(path, "w").write(content)
else:
    print("  [SKIP] types.py: already patched")
PATCH2

# Patch 3: operations.py - CompoundInfo type checks
OPS=autogrow/operators/operations.py
if grep -q 'type(x) is list' "$OPS" 2>/dev/null; then
    sed -i "s/type(x) is list/not isinstance(x, str)/g" "$OPS"
    sed -i "s/type(x) == list/not isinstance(x, str)/g" "$OPS"
    echo "  [OK] operations.py: CompoundInfo type checks"
else
    echo "  [SKIP] operations.py: already patched"
fi

# Patch 4: NNScore placeholder files
mkdir -p autogrow/docking/scoring/nn_score_exe/nnscore1
mkdir -p autogrow/docking/scoring/nn_score_exe/nnscore2
[ ! -f autogrow/docking/scoring/nn_score_exe/nnscore1/NNScore.py ] && \
    echo '# NNScore1 placeholder' > autogrow/docking/scoring/nn_score_exe/nnscore1/NNScore.py
[ ! -f autogrow/docking/scoring/nn_score_exe/nnscore2/NNScore2.py ] && \
    echo '# NNScore2 placeholder' > autogrow/docking/scoring/nn_score_exe/nnscore2/NNScore2.py
echo "  [OK] NNScore placeholder files"

# Patch 5: prepare_ligand4.py - fix os.path.basename stripping input path
PREP_LIG=~/final_project/mgltools_x86_64Linux2_1.5.7/MGLToolsPckgs/AutoDockTools/Utilities24/prepare_ligand4.py
if grep -q 'ligand_filename = os.path.basename(a)' "$PREP_LIG" 2>/dev/null; then
    sed -i 's/ligand_filename = os.path.basename(a)/ligand_filename = a/' "$PREP_LIG"
    echo "  [OK] prepare_ligand4.py: fixed path stripping"
else
    echo "  [SKIP] prepare_ligand4.py: already patched"
fi

# Patch 6: generate_line_plot.py - division by zero guard
python3 << 'PATCH6'
path = "autogrow/plotting/generate_line_plot.py"
content = open(path).read()
old = "gen_affinity_average = gen_affinity_sum / num_lines_counter"
new = "gen_affinity_average = gen_affinity_sum / num_lines_counter if num_lines_counter > 0 else 0.0"
if old in content and "if num_lines_counter > 0" not in content:
    content = content.replace(old, new, 1)
    open(path, "w").write(content)
    print("  [OK] generate_line_plot.py: division by zero guard")
else:
    print("  [SKIP] generate_line_plot.py: already patched")
PATCH6

# Patch 7: execute_scoring_mol.py - list to CompoundInfo conversion
python3 << 'PATCH7'
path = "autogrow/docking/scoring/execute_scoring_mol.py"
content = open(path).read()
if "isinstance(lig, list)" in content:
    print("  [SKIP] execute_scoring_mol.py: already patched")
else:
    old = """    lig_dict: Dict[str, CompoundInfo] = {}
    for lig in list_of_list_of_lig_data:
        if lig is None:
            continue
        lig_short_id = lig.short_id"""
    new = """    lig_dict: Dict[str, CompoundInfo] = {}
    for lig in list_of_list_of_lig_data:
        if lig is None:
            continue
        if isinstance(lig, list):
            if len(lig) >= 3:
                score_val = float(lig[-1]) if len(lig) > 3 else None
                lig = CompoundInfo(
                    smiles=str(lig[0]),
                    name=str(lig[1]),
                    short_id=str(lig[2]),
                    docking_score=score_val,
                    fitness_score=score_val,
                )
            else:
                continue
        lig_short_id = lig.short_id"""
    if old in content:
        content = content.replace(old, new, 1)
        open(path, "w").write(content)
        print("  [OK] execute_scoring_mol.py: list->CompoundInfo conversion")
    else:
        print("  [WARN] execute_scoring_mol.py: pattern not found")
PATCH7

# Patch 8: ranking_mol.py - parse scores from ranked .smi files + remove pdb + fix get_chosen_mol_full_data_list
python3 << 'PATCH8'
path = "autogrow/docking/ranking/ranking_mol.py"
content = open(path).read()
changed = False

# 8a: Fix get_usable_format to parse docking/diversity scores from ranked .smi files
old_usable = """            compoundInfo = CompoundInfo(smiles=parts[0], name=parts[1])
            usable_list_of_smiles.append(compoundInfo)"""
new_usable = """            compoundInfo = CompoundInfo(smiles=parts[0], name=parts[1])
            # Parse scores if present in ranked .smi files
            if len(parts) >= 4:
                try:
                    compoundInfo.docking_score = float(parts[-2])
                except (ValueError, IndexError):
                    pass
                try:
                    compoundInfo.diversity_score = float(parts[-1])
                except (ValueError, IndexError):
                    pass
            elif len(parts) == 3:
                try:
                    compoundInfo.docking_score = float(parts[2])
                except (ValueError, IndexError):
                    pass
            usable_list_of_smiles.append(compoundInfo)"""
if old_usable in content:
    content = content.replace(old_usable, new_usable, 1)
    changed = True
    print("  [OK] ranking_mol.py: get_usable_format now parses scores")
elif "Parse scores if present" in content:
    print("  [SKIP] ranking_mol.py: get_usable_format already patched")
else:
    print("  [WARN] ranking_mol.py: get_usable_format pattern not found")

# 8b: Remove pdb.set_trace() if present
if "import pdb; pdb.set_trace()" in content:
    content = content.replace("    import pdb; pdb.set_trace()\n", "")
    content = content.replace("    import pdb; pdb.set_trace()", "")
    changed = True
    print("  [OK] ranking_mol.py: removed pdb.set_trace()")
else:
    print("  [SKIP] ranking_mol.py: no pdb.set_trace() found")

# 8c: Fix get_chosen_mol_full_data_list to use CompoundInfo attributes
old_sorted = "    sorted_list = sorted(usable_list_of_smiles, key=lambda x: float(x[-2]))"
new_sorted = "    sorted_list = sorted(usable_list_of_smiles, key=lambda x: x.docking_score if x.docking_score is not None else float('inf'))"
if old_sorted in content:
    content = content.replace(old_sorted, new_sorted, 1)
    changed = True
    print("  [OK] ranking_mol.py: fixed get_chosen_mol_full_data_list sort")
elif "x.docking_score if x.docking_score" in content:
    print("  [SKIP] ranking_mol.py: get_chosen_mol_full_data_list already patched")
else:
    print("  [WARN] ranking_mol.py: get_chosen_mol_full_data_list sort pattern not found")

if changed:
    open(path, "w").write(content)
PATCH8

# Patch 9: rank_selection.py - fix old list-style access after initial sort
python3 << 'PATCH9'
path = "autogrow/docking/ranking/selecting/rank_selection.py"
content = open(path).read()
changed = False

# Fix second sort that uses list indexing
old_sort2 = "    new_sorted_list = sorted(\n        new_sorted_list,\n        key=lambda x: float(x[column_idx_to_select]),\n        reverse=reverse_sort,\n    )"
if old_sort2 not in content:
    # Try without escaped newlines
    old_sort2 = """    new_sorted_list = sorted(
        new_sorted_list,
        key=lambda x: float(x[column_idx_to_select]),
        reverse=reverse_sort,
    )"""
if old_sort2 in content:
    new_sort2 = """    new_sorted_list = sorted(
        new_sorted_list,
        key=lambda x: x.score_by_index_lookup(column_idx_to_select),
        reverse=reverse_sort,
    )"""
    content = content.replace(old_sort2, new_sort2, 1)
    changed = True
    print("  [OK] rank_selection.py: fixed second sort")

# Fix list indexing x[0] -> x.smiles
old_unique = "    if len(list({x[0] for x in new_sorted_list})) >= number_to_chose:"
if old_unique in content:
    content = content.replace(old_unique, "    if len(list({x.smiles for x in new_sorted_list})) >= number_to_chose:")
    content = content.replace("            if mol_info[0] in smiles_list:", "            if mol_info.smiles in smiles_list:")
    content = content.replace("            smiles_list.append(mol_info[0])", "            smiles_list.append(mol_info.smiles)")
    changed = True
    print("  [OK] rank_selection.py: fixed list indexing to use .smiles")

# Fix return value - smiles[0] -> smiles.smiles
old_return = "        top_choice_smile_order.append(smiles[0])"
if old_return in content:
    content = content.replace(old_return, "        top_choice_smile_order.append(smiles.smiles)")
    changed = True
    print("  [OK] rank_selection.py: fixed return value")

if changed:
    open(path, "w").write(content)
else:
    print("  [SKIP] rank_selection.py: already patched")
PATCH9

# Patch 10: execute_scoring_mol.py - graceful None handling when scoring fails
python3 << 'PATCH10'
path = "autogrow/docking/scoring/execute_scoring_mol.py"
content = open(path).read()

old = """    list_of_lig_data = scoring_object.run_scoring(file_path)
    if rescore_lig_efficiency:
        assert (
            lig_efficiency_scoring_object is not None
        ), "lig_efficiency_scoring_object is None"
        assert list_of_lig_data is not None, "list_of_lig_data is None"
        list_of_lig_data = lig_efficiency_scoring_object.get_lig_efficiency_rescore_from_a_file(
            file_path, list_of_lig_data
        )

    return list_of_lig_data"""

new = """    list_of_lig_data = scoring_object.run_scoring(file_path)
    if list_of_lig_data is None:
        return None
    if rescore_lig_efficiency:
        if lig_efficiency_scoring_object is None:
            return list_of_lig_data
        list_of_lig_data = lig_efficiency_scoring_object.get_lig_efficiency_rescore_from_a_file(
            file_path, list_of_lig_data
        )

    return list_of_lig_data"""

if old in content:
    content = content.replace(old, new, 1)
    open(path, "w").write(content)
    print("  [OK] execute_scoring_mol.py: graceful None handling for failed scoring")
elif "list_of_lig_data is None:" in content and "return None" in content:
    print("  [SKIP] execute_scoring_mol.py: already patched")
else:
    # Fallback: just replace the assert line
    if 'assert list_of_lig_data is not None' in content:
        content = content.replace(
            '        assert list_of_lig_data is not None, "list_of_lig_data is None"',
            '        if list_of_lig_data is None:\n            return None'
        )
        open(path, "w").write(content)
        print("  [OK] execute_scoring_mol.py: replaced assert with return None")
    else:
        print("  [SKIP] execute_scoring_mol.py: no assert found")
PATCH10

echo ""
echo "All patches applied successfully!"


# Patch 13: vina_docking.py - fix score crash when sorting compounds with no docking score
VINA_DOCK=autogrow/docking/docking_class/docking_class_children/vina_docking.py
if grep -q "Filter out compounds that failed docking" "$VINA_DOCK" 2>/dev/null; then
    echo "Patch 13: vina_docking.py sort fix — already applied, skipping."
else
    if grep -q 'smiles_list.sort(key=lambda x: x.score, reverse=False)' "$VINA_DOCK"; then
        sed -i 's|smiles_list.sort(key=lambda x: x.score, reverse=False)|# Filter out compounds that failed docking (no score)\n        smiles_list = [x for x in smiles_list if x.docking_score is not None]\n        if not smiles_list:\n            print("WARNING: No successfully docked molecules in this generation!")\n            output_ranked_smile_file = smile_file.replace(".smi", "") + "_ranked.smi"\n            with open(output_ranked_smile_file, "w") as output:\n                pass\n            return output_ranked_smile_file\n        smiles_list.sort(key=lambda x: x.docking_score if x.docking_score is not None else float(\x27inf\x27), reverse=False)|' "$VINA_DOCK"
        echo "Patch 13: vina_docking.py — fixed score crash in rank_and_save_output_smi"
    else
        echo "Patch 13: WARNING — target string not found in $VINA_DOCK"
    fi
fi

# Patch 14: types.py - score property returns inf instead of raising ValueError
TYPES=autogrow/types.py
if grep -q 'return float("inf")' "$TYPES" 2>/dev/null; then
    echo "Patch 14: types.py score property — already patched, skipping."
else
    if grep -q 'raise ValueError("No score available")' "$TYPES"; then
        sed -i 's|raise ValueError("No score available")|return float("inf")  # No score = worst possible|g' "$TYPES"
        echo "Patch 14: types.py — score property now returns inf instead of raising ValueError"
    else
        echo "Patch 14: WARNING — target string not found in $TYPES"
    fi
fi



# Patch 15: Install custom Lipinski filter with configurable thresholds
CUSTOM_FILTER=autogrow/operators/filter/filter_classes/filter_children_classes/custom_lipinski_filter.py
if [ -f "$CUSTOM_FILTER" ]; then
    echo "Patch 15: custom_lipinski_filter.py — already exists, skipping."
else
    cat > "$CUSTOM_FILTER" << 'CUSTOMEOF'
"""Custom Lipinski Filter with configurable thresholds."""
import __future__
import json
import os
import rdkit
import rdkit.Chem as Chem
import rdkit.Chem.Lipinski as Lipinski
import rdkit.Chem.Crippen as Crippen
import rdkit.Chem.Descriptors as Descriptors
rdkit.RDLogger.DisableLog("rdApp.*")
from autogrow.operators.filter.filter_classes.parent_filter_class import ParentFilter

_DEFAULTS = {"max_mw": 500.0, "max_logp": 5.0, "min_logp": -999.0, "max_h_donors": 5, "max_h_acceptors": 10}

def _load_config():
    config_path = os.path.expanduser("~/final_project/custom_filter_config.json")
    config = dict(_DEFAULTS)
    try:
        if os.path.exists(config_path):
            with open(config_path) as f:
                config.update(json.load(f))
    except Exception:
        pass
    return config

class CustomLipinskiFilter(ParentFilter):
    def run_filter(self, mol):
        config = _load_config()
        if Descriptors.ExactMolWt(mol) > config["max_mw"]:
            return False
        logp = Crippen.MolLogP(mol)
        if logp > config["max_logp"] or logp < config.get("min_logp", -999):
            return False
        if Lipinski.NumHDonors(mol) > config["max_h_donors"]:
            return False
        if Lipinski.NumHAcceptors(mol) > config["max_h_acceptors"]:
            return False
        return True
CUSTOMEOF
    echo "Patch 15: custom_lipinski_filter.py — created"
fi

# Patch 16: Register CustomLipinskiFilter in config_filters.py
CONFIG_FILTERS=autogrow/config/config_filters.py
if grep -q "CustomLipinskiFilter" "$CONFIG_FILTERS" 2>/dev/null; then
    echo "Patch 16: config_filters.py CustomLipinskiFilter — already registered, skipping."
else
    cd ~/final_project && python3 -c "
f = open(\"$PWD/autogrow4/$CONFIG_FILTERS\")
c = f.read()
f.close()
old = '    if \"BRENKFilter\" in vars_keys:\n        if params[\"BRENKFilter\"] is True:\n            filter_list.append(\"BRENKFilter\")\n    else:\n        params[\"BRENKFilter\"] = False'
new = old + '\n\n    if \"CustomLipinskiFilter\" in vars_keys:\n        if params[\"CustomLipinskiFilter\"] is True:\n            filter_list.append(\"CustomLipinskiFilter\")\n    else:\n        params[\"CustomLipinskiFilter\"] = False'
if old in c:
    c = c.replace(old, new)
    f = open(\"$PWD/autogrow4/$CONFIG_FILTERS\", \"w\")
    f.write(c)
    f.close()
    print(\"Patch 16: config_filters.py — added CustomLipinskiFilter\")
else:
    print(\"Patch 16: WARNING — could not find BRENKFilter block\")
" 2>&1
    cd ~/final_project/autogrow4
fi




# Patch 17: roulette_selection.py - ZeroDivisionError when diversity_score is 0.0
python3 << 'PATCH17'
path = "autogrow/docking/ranking/selecting/roulette_selection.py"
content = open(path).read()
old = "adjusted = [(x ** -2) for x in weight_scores]"
new = "adjusted = [((x + 1e-9) ** -2) for x in weight_scores]"
if old in content:
    content = content.replace(old, new, 1)
    open(path, "w").write(content)
    print("  [OK] roulette_selection.py: fixed ZeroDivisionError (epsilon guard)")
elif "1e-9" in content:
    print("  [SKIP] roulette_selection.py: already patched")
else:
    print("  [WARN] roulette_selection.py: pattern not found")
PATCH17

echo "=== All patches applied ==="


=== Applying AutoGrow4 compatibility patches ===
  [OK] MyMol.py: clean chirality patch
  [SKIP] types.py: already patched
  [SKIP] operations.py: already patched
  [OK] NNScore placeholder files
  [SKIP] prepare_ligand4.py: already patched
  [SKIP] generate_line_plot.py: already patched
  [SKIP] execute_scoring_mol.py: already patched
  [SKIP] ranking_mol.py: get_usable_format already patched
  [SKIP] ranking_mol.py: no pdb.set_trace() found
  [SKIP] ranking_mol.py: get_chosen_mol_full_data_list already patched
  [SKIP] rank_selection.py: already patched
  [SKIP] execute_scoring_mol.py: already patched

All patches applied successfully!
Patch 13: vina_docking.py sort fix — already applied, skipping.
Patch 14: types.py score property — already patched, skipping.
Patch 15: custom_lipinski_filter.py — already exists, skipping.
Patch 16: config_filters.py CustomLipinskiFilter — already registered, skipping.
  [SKIP] roulette_selection.py: already patched
=== All patches applied ===


In [12]:
# Fix nglview frontend version mismatch on SDSC Expanse
# Must patch the source file BEFORE importing nglview, since traitlets
# are set at class definition time and cannot be overridden at runtime.
import warnings, os, re, subprocess, sys
warnings.filterwarnings("ignore", message=".*pkg_resources.*")

# Step 1: Detect what JupyterLab has installed
_lab_version = None
try:
    _r = subprocess.run(['jupyter', 'labextension', 'list'],
                       capture_output=True, text=True, timeout=10)
    for line in (_r.stdout + _r.stderr).split('\n'):
        if 'nglview-js-widgets' in line:
            m = re.search(r'v(\d+\.\d+\.\d+)', line)
            if m:
                _lab_version = m.group(1)
                break
except Exception:
    pass

if _lab_version:
    print(f'JupyterLab nglview-js-widgets version: {_lab_version}')
    
    # Step 2: Find and patch nglview _frontend.py on disk BEFORE importing
    for path in sys.path:
        frontend_file = os.path.join(path, 'nglview', '_frontend.py')
        if os.path.exists(frontend_file):
            with open(frontend_file) as f:
                content = f.read()
            m = re.search(r"__frontend_version__\s*=\s*'([^']+)'", content)
            if m and m.group(1) != _lab_version:
                old_ver = m.group(1)
                new_content = content.replace(
                    f"__frontend_version__ = '{old_ver}'",
                    f"__frontend_version__ = '{_lab_version}'"
                )
                with open(frontend_file, 'w') as f:
                    f.write(new_content)
                print(f'Patched {frontend_file}: {old_ver} -> {_lab_version}')
            elif m:
                print(f'nglview _frontend.py already matches ({m.group(1)})')
            break

# Step 3: Now import nglview (will use the patched version)
try:
    import nglview
    print(f'nglview loaded successfully (version {nglview.__version__})')
except Exception as e:
    print(f'nglview import issue: {e}')
    print('Scoring and results viewing will still work.')

JupyterLab nglview-js-widgets version: 3.1.5
nglview _frontend.py already matches (3.1.5)
nglview loaded successfully (version 0.0.0)


### 1.4b Download GID1 Receptor (PDB 2ZSH & 3ED1)

AutoGrow4 will be tested with the **GID1 (Gibberellin Insensitive Dwarf 1)** receptor.

In [6]:
%%bash
cd ~/final_project/receptors 2>/dev/null || mkdir -p ~/final_project/receptors && cd ~/final_project/receptors

# Download GID1 receptor structures from RCSB PDB
if [ ! -f "2ZSH.pdb" ]; then
    echo "Downloading 2ZSH (GID1 with GA3 + GID2)..."
    wget -q "https://files.rcsb.org/download/2ZSH.pdb" -O 2ZSH.pdb
    echo "  Downloaded: $(wc -l < 2ZSH.pdb) lines"
else
    echo "2ZSH.pdb already present."
fi

if [ ! -f "3ED1.pdb" ]; then
    echo "Downloading 3ED1 (GID1 with GA4)..."
    wget -q "https://files.rcsb.org/download/3ED1.pdb" -O 3ED1.pdb
    echo "  Downloaded: $(wc -l < 3ED1.pdb) lines"
else
    echo "3ED1.pdb already present."
fi

echo ""
echo "Receptor files:"
ls -la ~/final_project/receptors/*.pdb

# Convert PDB to PDBQT using MGLTools (required for AutoGrow4/Vina docking)
# Note: Do NOT use obabel for receptor conversion - AutoGrow4's MGLTools can't parse obabel PDBQT format
MGLTOOLS_BIN=~/final_project/mgltools_x86_64Linux2_1.5.7
for pdb in ~/final_project/receptors/*.pdb; do
    pdbqt="${pdb%.pdb}.pdbqt"
    if [ ! -f "$pdbqt" ]; then
        echo "Converting $(basename $pdb) to PDBQT via MGLTools..."
        $MGLTOOLS_BIN/bin/pythonsh $MGLTOOLS_BIN/MGLToolsPckgs/AutoDockTools/Utilities24/prepare_receptor4.py \
            -r "$pdb" -o "$pdbqt" -A hydrogens 2>&1
        echo "  Result: $(wc -l < "$pdbqt") lines"
    else
        echo "$(basename $pdbqt) already present."
    fi
done

2ZSH.pdb already present.
3ED1.pdb already present.

Receptor files:
-rw-r--r-- 1 mschnur iit130  316062 Mar 13 19:34 /home/mschnur/final_project/receptors/2ZSH.pdb
-rw-r--r-- 1 mschnur iit130 1427139 Mar 13 19:34 /home/mschnur/final_project/receptors/3ED1.pdb
2ZSH.pdbqt already present.
3ED1.pdbqt already present.


### 1.5 Install Force Fields (CHARMM, AMBER, OpenFF)

These force fields are used for ligand optimization and binding-site analysis.

In [ ]:
%%bash
# Install OpenMM and force fields
conda install -y -c conda-forge openmm openmmforcefields 2>&1 | tail -5
echo "--- OpenMM + CHARMM/AMBER force fields (openmmforcefields) installed ---"

# Install OpenFF Toolkit + Sage force field
conda install -y -c conda-forge openff-toolkit openff-forcefields 2>&1 | tail -5
echo "--- OpenFF Toolkit + Sage force field installed ---"

# Install OpenFFCHARMM (ParmEd-based CHARMM compatibility)
pip install openmm-forcefields 2>&1 | tail -3
echo "--- OpenFF/CHARMM bridge installed ---"

echo "\n=== All force fields installed ==="

### 1.6 Verify Installation

In [13]:
import importlib

deps = {
    "rdkit": "RDKit",
    "numpy": "NumPy",
    "scipy": "SciPy",
    "matplotlib": "Matplotlib",
    "func_timeout": "func_timeout",
    "mpi4py": "mpi4py",
    "nglview": "NGLView",
    "ipywidgets": "ipywidgets",
    "openmm": "OpenMM",
    "openmmforcefields": "openmmforcefields (CHARMM/AMBER)",
    "openff.toolkit": "OpenFF Toolkit",
}

print("Dependency Check:")
print("=" * 50)
all_ok = True
for module, name in deps.items():
    try:
        m = importlib.import_module(module)
        ver = getattr(m, "__version__", "OK")
        print(f"  [OK] {name}: {ver}")
    except ImportError:
        print(f"  [!!] {name}: NOT INSTALLED")
        all_ok = False

# Check obabel
import shutil
obabel = shutil.which("obabel")
print(f"  [{'OK' if obabel else '!!'}] OpenBabel: {obabel or 'NOT FOUND'}")

# Check MGLTools
mgl = os.path.exists(os.path.join(MGLTOOLS_DIR, "bin", "pythonsh"))
print(f"  [{'OK' if mgl else '!!'}] MGLTools: {MGLTOOLS_DIR if mgl else 'NOT FOUND'}")

# Check AutoGrow4
ag4 = os.path.exists(os.path.join(AUTOGROW_DIR, "run_autogrow.py"))
print(f"  [{'OK' if ag4 else '!!'}] AutoGrow4: {AUTOGROW_DIR if ag4 else 'NOT FOUND'}")

print("=" * 50)
if all_ok and obabel and mgl and ag4:
    print("All dependencies verified!")
else:
    print("Some dependencies are missing. Please check above.")

Dependency Check:
  [OK] RDKit: 2025.03.6
  [OK] NumPy: 2.2.6
  [OK] SciPy: 1.15.2
  [OK] Matplotlib: 3.10.8
  [OK] func_timeout: 4.3.5
  [OK] mpi4py: 4.1.1
  [OK] NGLView: 0.0.0
  [OK] ipywidgets: 8.1.8
  [OK] OpenMM: 8.2
  [OK] openmmforcefields (CHARMM/AMBER): 0.15.1
  [OK] OpenFF Toolkit: 0.16.10
  [OK] OpenBabel: /home/mschnur/miniconda3/envs/visualization/bin/obabel
  [OK] MGLTools: /home/mschnur/final_project/mgltools_x86_64Linux2_1.5.7
  [OK] AutoGrow4: /home/mschnur/final_project/autogrow4
All dependencies verified!


---

## 2. Receptor Visualization & Binding Box Definition

Use the **NGL Viewer** below to visualize your receptor and interactively define the docking box.  
**Red** (X), **Green** (Y), and **Blue** (Z) arrows show the box edges from the corner.  
A **black sphere** marks the box center. Adjust the sliders to position the box around the binding site.

The box center and size values will be used by AutoGrow4 for docking.

In [14]:
import nglview as nv
import ipywidgets as widgets
import time

# ══════════════════════════════════════════════════════════════════
# Load Receptor Structure
# ══════════════════════════════════════════════════════════════════

receptor_pdb = os.path.join(RECEPTOR_DIR, "2ZSH.pdb")
receptor_pdbqt = os.path.join(RECEPTOR_DIR, "2ZSH.pdbqt")

def _pdbqt_to_pdb_text(filepath):
    """Convert PDBQT to PDB text for nglview."""
    pdb_lines = []
    with open(filepath) as f:
        for line in f:
            record = line[:6].strip()
            if record in ('ROOT', 'ENDROOT', 'BRANCH', 'ENDBRANCH', 'TORSDOF'):
                continue
            if record in ('ATOM', 'HETATM'):
                pdb_lines.append(line[:66].rstrip())
            elif record in ('TER', 'END', 'MODEL', 'ENDMDL', 'REMARK', 'HEADER', 'TITLE', 'CONECT'):
                pdb_lines.append(line.rstrip())
    return '\n'.join(pdb_lines)

# Load receptor text
if os.path.exists(receptor_pdb):
    with open(receptor_pdb) as f:
        _rec_text = f.read()
    print(f"Loaded receptor: {receptor_pdb}")
elif os.path.exists(receptor_pdbqt):
    _rec_text = _pdbqt_to_pdb_text(receptor_pdbqt)
    print(f"Loaded receptor (from PDBQT): {receptor_pdbqt}")
else:
    _rec_text = None
    print("WARNING: No receptor file found.")

# Create NGL viewer
view = nv.NGLWidget()
if _rec_text:
    view.add_component(nv.TextStructure(_rec_text, ext="pdb"))
    time.sleep(0.5)

    view.clear()
    view.add_cartoon(selection="protein", color="sstruc", opacity=0.5, component=0)

    # ══════════════════════════════════════════════════════════════════
    # Binding Site Residues (2ZSH PDB numbering)
    # PDB offset: subtract 7 from literature/UniProt residue numbers
    # ══════════════════════════════════════════════════════════════════
    # H-Bond:       Ser116->109, Tyr127->120, Phe238->231, Arg241->234, Ser198->191
    # Hydrophobic:  Ile24->17, Phe245->238, Tyr247->240, Tyr254->247
    # Selectivity:  Ile133->126, Leu330->323
    # Lid/Enclosure: Leu18->11, Trp21->14, Phe27->20, Leu29->22

    _hbond_res = ["109", "120", "231", "234", "191"]
    _hydro_res = ["17", "238", "240", "247"]
    _select_res = ["126", "323"]
    _lid_res = ["11", "14", "20", "22"]
    _all_res = _hbond_res + _hydro_res + _select_res + _lid_res

    # Color-coded residue display
    _hb_sel = "(" + " or ".join(_hbond_res) + ") and sidechainAttached"
    view.add_licorice(selection=_hb_sel, color="#2166ac", component=0)

    _hy_sel = "(" + " or ".join(_hydro_res) + ") and sidechainAttached"
    view.add_licorice(selection=_hy_sel, color="#d95f02", component=0)

    _sl_sel = "(" + " or ".join(_select_res) + ") and sidechainAttached"
    view.add_licorice(selection=_sl_sel, color="#1b7837", component=0)

    _ld_sel = "(" + " or ".join(_lid_res) + ") and sidechainAttached"
    view.add_licorice(selection=_ld_sel, color="#ae017e", component=0)

    # Labels for all binding site residues
    _label_sel = "(" + " or ".join(_all_res) + ") and .CA"
    view.add_label(selection=_label_sel, color="white",
                   labelType="format", labelFormat="%(resname)s%(resno)s",
                   labelGrouping="residue", component=0,
                   zOffset=2.0, fixedSize=True,
                   attachment="middle_center", showBackground=True,
                   backgroundColor="black", backgroundOpacity=0.6)

    # Show co-crystallized ligand (GA3) as ball-and-stick
    view.add_ball_and_stick(selection="hetero and not water and not ion",
                            color="element", component=0)

    print("Binding site residues (PDB numbering):")
    print(f"  H-Bond (blue):       Ser109, Tyr120, Phe231, Arg234, Ser191")
    print(f"  Hydrophobic (orange): Ile17, Phe238, Tyr240, Tyr247")
    print(f"  Selectivity (green):  Ile126, Leu323")
    print(f"  Lid/Enclosure (pink): Leu11, Trp14, Phe20, Leu22")
    print(f"  Ligand: ball-and-stick (hetero atoms)")

# ══════════════════════════════════════════════════════════════════
# Interactive Binding Box Sliders
# ══════════════════════════════════════════════════════════════════

binding_box = {
    "center_x": 51.0, "center_y": 59.5, "center_z": 37.4,
    "size_x": 17.0, "size_y": 18.0, "size_z": 18.0
}

c_x = widgets.FloatSlider(value=51.0, min=0, max=100, step=0.5, description='X:')
c_y = widgets.FloatSlider(value=59.5, min=0, max=100, step=0.5, description='Y:')
c_z = widgets.FloatSlider(value=37.4, min=0, max=100, step=0.5, description='Z:')
s_x = widgets.FloatSlider(value=17.0, min=5, max=40, step=0.5, description='X:')
s_y = widgets.FloatSlider(value=18.0, min=5, max=40, step=0.5, description='Y:')
s_z = widgets.FloatSlider(value=18.0, min=5, max=40, step=0.5, description='Z:')

def update_box(cx, cy, cz, sx, sy, sz):
    # Remove previous shape components (everything after component 0 = protein)
    while view.n_components > 1:
        view.remove_component(view.n_components - 1)

    hx, hy, hz = sx/2, sy/2, sz/2
    c0 = [cx-hx, cy-hy, cz-hz]
    c1 = [cx+hx, cy-hy, cz-hz]
    c2 = [cx+hx, cy+hy, cz-hz]
    c3 = [cx-hx, cy+hy, cz-hz]
    c4 = [cx-hx, cy-hy, cz+hz]
    c5 = [cx+hx, cy-hy, cz+hz]
    c6 = [cx+hx, cy+hy, cz+hz]
    c7 = [cx-hx, cy+hy, cz+hz]

    box_col = [0.3, 0.7, 0.9]
    for a, b in [(c0,c1),(c1,c2),(c2,c3),(c3,c0),
                 (c4,c5),(c5,c6),(c6,c7),(c7,c4),
                 (c0,c4),(c1,c5),(c2,c6),(c3,c7)]:
        view.shape.add_cylinder(a, b, box_col, 0.12)

    # Axis arrows from one corner
    view.shape.add_arrow(c0, [c0[0]+sx, c0[1], c0[2]], [1, 0, 0], 0.25)
    view.shape.add_arrow(c0, [c0[0], c0[1]+sy, c0[2]], [0, 1, 0], 0.25)
    view.shape.add_arrow(c0, [c0[0], c0[1], c0[2]+sz], [0, 0, 1], 0.25)

    # Center sphere
    view.shape.add_sphere([cx, cy, cz], [0, 0, 0], 0.4)

    global binding_box
    binding_box = {
        "center_x": cx, "center_y": cy, "center_z": cz,
        "size_x": sx, "size_y": sy, "size_z": sz
    }

out = widgets.interactive_output(update_box, {
    'cx': c_x, 'cy': c_y, 'cz': c_z,
    'sx': s_x, 'sy': s_y, 'sz': s_z
})

center_ui = widgets.VBox([widgets.Label("Box Center"), c_x, c_y, c_z])
size_ui = widgets.VBox([widgets.Label("Box Size (A)"), s_x, s_y, s_z])
legend = widgets.HTML(
    "<div style='font-size:12px; padding:8px; border:1px solid #ccc; border-radius:4px;'>"
    "<b>Residue Legend:</b><br>"
    "<span style='color:#2166ac;font-size:16px;'>&#9632;</span> H-Bond: Ser109, Tyr120, Phe231, Arg234, Ser191<br>"
    "<span style='color:#d95f02;font-size:16px;'>&#9632;</span> Hydrophobic: Ile17, Phe238, Tyr240, Tyr247<br>"
    "<span style='color:#1b7837;font-size:16px;'>&#9632;</span> Selectivity: Ile126, Leu323<br>"
    "<span style='color:#ae017e;font-size:16px;'>&#9632;</span> Lid/Enclosure: Leu11, Trp14, Phe20, Leu22<br>"
    "<br><b>Box:</b> Cyan wireframe with "
    "<span style='color:red;'>X</span> "
    "<span style='color:green;'>Y</span> "
    "<span style='color:blue;'>Z</span> arrows"
    "</div>"
)
ui = widgets.HBox([center_ui, size_ui, widgets.VBox([legend])])

widgets.VBox([view, ui])


Loaded receptor: /home/mschnur/final_project/receptors/2ZSH.pdb
Binding site residues: 115, 116, 191, 289, 319 (PDB numbering)
Gly122->115, Ser123->116, Ser198->191, Asp296->289, Val326->319


---

## 3. AutoGrow4 Parameter Configuration (GUI)

Configure all AutoGrow4 parameters using the tabbed interface below.  
Changes are reflected in the Slurm submission script automatically.

In [15]:
import ipywidgets as widgets
from IPython.display import display
import json
import os

# ══════════════════════════════════════════════════════════════
# TAB 1: Input Files & Paths
# ══════════════════════════════════════════════════════════════

w_receptor = widgets.Text(
    value=os.path.join(RECEPTOR_DIR, "2ZSH.pdbqt"),
    description="Receptor PDB:",
    layout=widgets.Layout(width="95%"),
    style={"description_width": "160px"}
)
# Scan source_compounds directory for available .smi databases
_source_compounds_dir = os.path.join(AUTOGROW_DIR, "source_compounds")
_smi_files = sorted([f for f in os.listdir(_source_compounds_dir) if f.endswith(".smi")]) if os.path.isdir(_source_compounds_dir) else ["naphthalene_smiles.smi"]

w_source_db = widgets.SelectMultiple(
    options=_smi_files,
    value=["naphthalene_smiles.smi"] if "naphthalene_smiles.smi" in _smi_files else [_smi_files[0]],
    description="Source DB:",
    layout=widgets.Layout(width="95%", height="120px"),
    style={"description_width": "160px"}
)
w_source_db_label = widgets.HTML(
    value="<small>Hold Ctrl/Cmd to select multiple databases. They will be merged into one file.</small>"
)
_refresh_db_btn = widgets.Button(description="Refresh Database List", button_style="info",
    layout=widgets.Layout(width="200px"), icon="refresh")
def _refresh_db_list(b):
    """Rescan source_compounds directory for new .smi files."""
    new_files = sorted([f for f in os.listdir(_source_compounds_dir) if f.endswith(".smi")]) if os.path.isdir(_source_compounds_dir) else []
    if new_files:
        w_source_db.options = new_files
        w_source_db_label.value = f"<small>Found {len(new_files)} databases. Hold Ctrl/Cmd to select multiple.</small>"
    else:
        w_source_db_label.value = "<small>No .smi files found in source_compounds directory.</small>"
_refresh_db_btn.on_click(_refresh_db_list)

# Hidden text field updated by dropdown selection
w_source_compounds = widgets.Text(
    value=os.path.join(AUTOGROW_DIR, "source_compounds", "naphthalene_smiles.smi"),
    description="Source File:",
    layout=widgets.Layout(width="95%"),
    style={"description_width": "160px"}
)

def _on_source_db_change(change):
    selected = list(change["new"])
    if len(selected) == 1:
        w_source_compounds.value = os.path.join(_source_compounds_dir, selected[0])
    elif len(selected) > 1:
        # Merge selected files into a combined .smi file
        merged_path = os.path.join(_source_compounds_dir, "merged_selection.smi")
        lines_set = set()
        for smi_file in selected:
            fpath = os.path.join(_source_compounds_dir, smi_file)
            if os.path.exists(fpath):
                with open(fpath) as f:
                    for line in f:
                        line = line.strip()
                        if line:
                            lines_set.add(line)
        with open(merged_path, "w") as f:
            f.write("\n".join(sorted(lines_set)) + "\n")
        w_source_compounds.value = merged_path

w_source_db.observe(_on_source_db_change, names="value")
w_output_folder = widgets.Text(
    value=OUTPUT_DIR,
    description="Output Folder:",
    layout=widgets.Layout(width="95%"),
    style={"description_width": "160px"}
)
w_mgltools_dir = widgets.Text(
    value=MGLTOOLS_DIR,
    description="MGLTools Dir:",
    layout=widgets.Layout(width="95%"),
    style={"description_width": "160px"}
)
w_obabel_path = widgets.Text(
    value="",
    description="obabel path:",
    placeholder="Leave blank to use MGLTools instead",
    layout=widgets.Layout(width="95%"),
    style={"description_width": "160px"}
)
w_conversion = widgets.Dropdown(
    options=["MGLToolsConversion", "ObabelConversion"],
    value="MGLToolsConversion",
    description="PDB Converter:",
    style={"description_width": "160px"}
)

tab_files = widgets.VBox([
    widgets.HTML("<h4>Input Files & Paths</h4>"),
    w_receptor, w_source_db, widgets.HBox([w_source_db_label, _refresh_db_btn]), w_source_compounds, w_output_folder,
    widgets.HTML("<hr>"),
    widgets.HTML("<b>PDB → PDBQT Conversion Tool</b>"),
    w_conversion, w_mgltools_dir, w_obabel_path,
])

# ══════════════════════════════════════════════════════════════
# TAB 2: Evolutionary Algorithm Settings
# ══════════════════════════════════════════════════════════════

w_num_gens = widgets.IntSlider(value=5, min=1, max=100, description="Generations:", style={"description_width": "160px"}, layout=widgets.Layout(width="70%"))

# First generation
w_mutants_first = widgets.IntText(value=50, description="Mutants (1st gen):", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))
w_crossovers_first = widgets.IntText(value=50, description="Crossovers (1st gen):", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))
w_elitism_first = widgets.IntText(value=10, description="Elitism (1st gen):", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))

# Subsequent generations
w_mutants = widgets.IntText(value=50, description="Mutants (per gen):", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))
w_crossovers = widgets.IntText(value=50, description="Crossovers (per gen):", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))
w_elitism = widgets.IntText(value=50, description="Elitism (per gen):", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))

# Seeding
w_top_seed = widgets.IntText(value=50, description="Top mols to seed:", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))
w_diversity_seed = widgets.IntText(value=10, description="Diversity seed (1st):", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))
w_diversity_deprec = widgets.IntText(value=2, description="Diversity depreciation:", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))

# Selection
w_selector = widgets.Dropdown(
    options=["Rank_Selector", "Roulette_Selector", "Tournament_Selector"],
    value="Rank_Selector",
    description="Selector:",
    style={"description_width": "160px"}
)
w_tourn_size = widgets.FloatText(value=0.1, description="Tournament size:", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))

# Mutation proportions
w_prop_mut_extending = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description="Mutations extending:", style={"description_width": "160px"}, layout=widgets.Layout(width="50%"), readout_format=".2f")
w_prop_mut_deleting = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description="Mutations deleting:", style={"description_width": "160px"}, layout=widgets.Layout(width="50%"), readout_format=".2f")
w_max_attempts = widgets.IntText(value=10, description="Max gen attempts:", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))

# Reaction library
w_rxn_library = widgets.Dropdown(
    options=["all_rxns", "click_chem_rxns", "robust_rxns", "agro_rxns (Agricultural)", "Custom"],
    value="all_rxns",
    description="Reaction Library:",
    style={"description_width": "160px"}
)

# Options
w_start_new = widgets.Checkbox(value=True, description="Start a new run")
w_filter_source = widgets.Checkbox(value=True, description="Filter source compounds")
w_use_docked = widgets.Checkbox(value=True, description="Dock source compounds")
w_redock_elite = widgets.Checkbox(value=False, description="Redock elite from prev gen")
w_generate_plot = widgets.Checkbox(value=True, description="Generate fitness plot")
w_reduce_files = widgets.Checkbox(value=True, description="Reduce file sizes")

tab_evolution = widgets.VBox([
    widgets.HTML("<h4>Evolutionary Algorithm Settings</h4>"),
    w_num_gens,
    widgets.HTML("<b>First Generation</b>"),
    widgets.HBox([w_mutants_first, w_crossovers_first, w_elitism_first]),
    widgets.HTML("<b>Subsequent Generations</b>"),
    widgets.HBox([w_mutants, w_crossovers, w_elitism]),
    widgets.HTML("<hr><b>Seeding & Selection</b>"),
    widgets.HBox([w_top_seed, w_diversity_seed, w_diversity_deprec]),
    widgets.HBox([w_selector, w_tourn_size]),
    widgets.HTML("<hr><b>Mutation Proportions</b>"),
    widgets.HBox([w_prop_mut_extending, w_prop_mut_deleting, w_max_attempts]),
    widgets.HTML("<hr><b>Reaction Library</b>"),
    w_rxn_library,
    widgets.HTML("<hr><b>Options</b>"),
    widgets.HBox([w_start_new, w_filter_source, w_use_docked]),
    widgets.HBox([w_redock_elite, w_generate_plot, w_reduce_files]),
])

# ══════════════════════════════════════════════════════════════
# TAB 3: Filters
# ══════════════════════════════════════════════════════════════

w_lipinski_strict = widgets.Checkbox(value=False, description="Lipinski Strict")
w_lipinski_lenient = widgets.Checkbox(value=True, description="Lipinski Lenient")
w_ghose = widgets.Checkbox(value=False, description="Ghose")
w_ghose_modified = widgets.Checkbox(value=False, description="Ghose Modified")
w_mozziconacci = widgets.Checkbox(value=False, description="Mozziconacci")
w_vandewaterbeemd = widgets.Checkbox(value=False, description="Van de Waterbeemd")
w_pains = widgets.Checkbox(value=False, description="PAINS")
w_nih = widgets.Checkbox(value=False, description="NIH")
w_brenk = widgets.Checkbox(value=False, description="BRENK")
w_no_filters = widgets.Checkbox(value=False, description="No Filters (disable all)")

filter_info = widgets.HTML("""
<style>table.finfo td { padding: 2px 10px; font-size: 12px; }</style>
<table class="finfo">
<tr><td><b>Lipinski Strict</b></td><td>MW ≤ 500, LogP ≤ 5, H-donors ≤ 5, H-acceptors ≤ 10</td></tr>
<tr><td><b>Lipinski Lenient</b></td><td>Passes if meets 3 of 4 Lipinski criteria</td></tr>
<tr><td><b>Ghose</b></td><td>MW 160-480, LogP -0.4-5.6, atoms 20-70, refractivity 40-130</td></tr>
<tr><td><b>PAINS</b></td><td>Pan-assay interference compounds filter</td></tr>
<tr><td><b>NIH</b></td><td>NIH chemical genomics center substructure filter</td></tr>
<tr><td><b>BRENK</b></td><td>Brenk unwanted substructure filter</td></tr>
</table>
""")

# Custom Lipinski filter with configurable thresholds and per-parameter enable/disable
w_custom_lipinski = widgets.Checkbox(value=False, description="Custom Lipinski (configurable)")

# Per-parameter enable checkboxes and range inputs
w_mw_enable = widgets.Checkbox(value=True, description="Molecular Weight", layout=widgets.Layout(width="180px"))
w_mw_min = widgets.FloatText(value=0.0, description="Min:", style={"description_width": "35px"}, layout=widgets.Layout(width="130px"))
w_mw_max = widgets.FloatText(value=500.0, description="Max:", style={"description_width": "35px"}, layout=widgets.Layout(width="130px"))

w_logp_enable = widgets.Checkbox(value=True, description="LogP", layout=widgets.Layout(width="180px"))
w_logp_min = widgets.FloatText(value=-999.0, description="Min:", style={"description_width": "35px"}, layout=widgets.Layout(width="130px"))
w_logp_max = widgets.FloatText(value=5.0, description="Max:", style={"description_width": "35px"}, layout=widgets.Layout(width="130px"))

w_hdonors_enable = widgets.Checkbox(value=True, description="H-Bond Donors", layout=widgets.Layout(width="180px"))
w_hdonors_min = widgets.IntText(value=0, description="Min:", style={"description_width": "35px"}, layout=widgets.Layout(width="130px"))
w_hdonors_max = widgets.IntText(value=5, description="Max:", style={"description_width": "35px"}, layout=widgets.Layout(width="130px"))

w_hacceptors_enable = widgets.Checkbox(value=True, description="H-Bond Acceptors", layout=widgets.Layout(width="180px"))
w_hacceptors_min = widgets.IntText(value=0, description="Min:", style={"description_width": "35px"}, layout=widgets.Layout(width="130px"))
w_hacceptors_max = widgets.IntText(value=10, description="Max:", style={"description_width": "35px"}, layout=widgets.Layout(width="130px"))

w_rotbonds_enable = widgets.Checkbox(value=True, description="Rotatable Bonds", layout=widgets.Layout(width="180px"))
w_rotbonds_min = widgets.IntText(value=0, description="Min:", style={"description_width": "35px"}, layout=widgets.Layout(width="130px"))
w_rotbonds_max = widgets.IntText(value=10, description="Max:", style={"description_width": "35px"}, layout=widgets.Layout(width="130px"))

w_tpsa_enable = widgets.Checkbox(value=False, description="TPSA", layout=widgets.Layout(width="180px"))
w_tpsa_min = widgets.FloatText(value=0.0, description="Min:", style={"description_width": "35px"}, layout=widgets.Layout(width="130px"))
w_tpsa_max = widgets.FloatText(value=140.0, description="Max:", style={"description_width": "35px"}, layout=widgets.Layout(width="130px"))

w_aromrings_enable = widgets.Checkbox(value=False, description="Aromatic Rings", layout=widgets.Layout(width="180px"))
w_aromrings_min = widgets.IntText(value=0, description="Min:", style={"description_width": "35px"}, layout=widgets.Layout(width="130px"))
w_aromrings_max = widgets.IntText(value=4, description="Max:", style={"description_width": "35px"}, layout=widgets.Layout(width="130px"))

_custom_filter_box = widgets.VBox([
    widgets.HTML("<small><i>Uncheck a parameter to exclude it from filtering. Set min/max for ranges.</i></small>"),
    widgets.HBox([w_mw_enable, w_mw_min, w_mw_max]),
    widgets.HBox([w_logp_enable, w_logp_min, w_logp_max]),
    widgets.HBox([w_hdonors_enable, w_hdonors_min, w_hdonors_max]),
    widgets.HBox([w_hacceptors_enable, w_hacceptors_min, w_hacceptors_max]),
    widgets.HBox([w_rotbonds_enable, w_rotbonds_min, w_rotbonds_max]),
    widgets.HBox([w_tpsa_enable, w_tpsa_min, w_tpsa_max]),
    widgets.HBox([w_aromrings_enable, w_aromrings_min, w_aromrings_max]),
], layout=widgets.Layout(padding="5px 0 0 25px"))

def _toggle_custom_filter(change):
    _custom_filter_box.layout.display = "" if change["new"] else "none"
    if change["new"]:
        w_lipinski_strict.value = False
        w_lipinski_lenient.value = False

w_custom_lipinski.observe(_toggle_custom_filter, names="value")
_custom_filter_box.layout.display = "none"

tab_filters = widgets.VBox([
    widgets.HTML("<h4>Ligand Filters</h4>"),
    widgets.HTML("<p>Select which drug-likeness filters to apply to generated molecules:</p>"),
    widgets.HBox([
        widgets.VBox([w_lipinski_strict, w_lipinski_lenient, w_ghose, w_ghose_modified, w_mozziconacci]),
        widgets.VBox([w_vandewaterbeemd, w_pains, w_nih, w_brenk, w_no_filters]),
    ]),
    widgets.HTML("<hr><b>Custom Lipinski Filter (set your own thresholds)</b>"),
    w_custom_lipinski,
    _custom_filter_box,
    widgets.HTML("<hr>"),
    filter_info,
])

# ══════════════════════════════════════════════════════════════
# TAB 4: Docking & Scoring
# ══════════════════════════════════════════════════════════════

w_dock_choice = widgets.Dropdown(
    options=["VinaDocking", "QuickVina2Docking"],
    value="QuickVina2Docking",
    description="Docking Engine:",
    style={"description_width": "160px"}
)
w_exhaustiveness = widgets.IntSlider(value=8, min=1, max=32, description="Exhaustiveness:", style={"description_width": "160px"}, layout=widgets.Layout(width="60%"))
w_num_modes = widgets.IntSlider(value=9, min=1, max=20, description="Num Modes:", style={"description_width": "160px"}, layout=widgets.Layout(width="60%"))
w_dock_timeout = widgets.IntText(value=600, description="Dock Timeout (s):", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))

w_scoring = widgets.Dropdown(
    options=["VINA", "NN1", "NN2"],
    value="VINA",
    description="Scoring Function:",
    style={"description_width": "160px"}
)
w_lig_efficiency = widgets.Checkbox(value=True, description="Rescore with ligand efficiency")

scoring_info = widgets.HTML("""
<style>table.sinfo td { padding: 2px 10px; font-size: 12px; }</style>
<table class="sinfo">
<tr><td><b>VINA</b></td><td>Uses Autodock Vina's built-in scoring (primary docking score)</td></tr>
<tr><td><b>NN1 (NNScore1)</b></td><td>Neural-network rescoring (trained on Vina 1.1.2 outputs)</td></tr>
<tr><td><b>NN2 (NNScore2)</b></td><td>Updated neural-network rescoring with improved predictions</td></tr>
<tr><td><b>Ligand Efficiency</b></td><td>Score / number of heavy atoms (normalizes for molecule size)</td></tr>
</table>
<p><i>Note: NNScore1 and NNScore2 require Autodock Vina 1.1.2 as the docking engine.</i></p>
""")

tab_docking = widgets.VBox([
    widgets.HTML("<h4>Docking & Scoring</h4>"),
    w_dock_choice, w_exhaustiveness, w_num_modes, w_dock_timeout,
    widgets.HTML("<hr><b>Scoring / Rescoring</b>"),
    widgets.HBox([w_scoring, w_lig_efficiency]),
    widgets.HTML("<hr>"),
    scoring_info,
])

# ══════════════════════════════════════════════════════════════
# TAB 5: Force Fields
# ══════════════════════════════════════════════════════════════

w_ff_choice = widgets.SelectMultiple(
    options=["CHARMM (openmmforcefields)", "AMBER (openmmforcefields)", "OpenFF Sage 2.0", "SQM2.20 (Semi-empirical)"],
    value=["CHARMM (openmmforcefields)", "AMBER (openmmforcefields)"],
    description="Force Fields:",
    rows=4,
    style={"description_width": "160px"},
    layout=widgets.Layout(width="60%")
)

ff_info = widgets.HTML("""
<style>table.ffinfo td { padding: 2px 10px; font-size: 12px; }</style>
<table class="ffinfo">
<tr><td><b>CHARMM</b></td><td>Chemistry at HARvard Macromolecular Mechanics — general-purpose biomolecular FF</td></tr>
<tr><td><b>AMBER</b></td><td>Assisted Model Building with Energy Refinement — widely used for proteins/nucleic acids</td></tr>
<tr><td><b>OpenFF Sage</b></td><td>Open Force Field Initiative Sage 2.0 — modern, data-driven small-molecule FF</td></tr>
<tr><td><b>SQM2.20</b></td><td>Semi-empirical quantum mechanics method for energy refinement</td></tr>
</table>
<p><i>Force fields are available via openmmforcefields and openff-toolkit (installed above).<br>
They can be used for post-docking energy minimization and ligand optimization.</i></p>
""")

tab_forcefields = widgets.VBox([
    widgets.HTML("<h4>Force Fields</h4>"),
    widgets.HTML("<p>Select force fields available for ligand optimization (Ctrl+Click for multiple):</p>"),
    w_ff_choice,
    widgets.HTML("<hr>"),
    ff_info,
])

# ══════════════════════════════════════════════════════════════
# TAB 6: Gypsum-DL (3D Conversion)
# ══════════════════════════════════════════════════════════════

w_max_variants = widgets.IntSlider(value=3, min=1, max=10, description="Max variants/mol:", style={"description_width": "160px"}, layout=widgets.Layout(width="60%"))
w_gypsum_thoroughness = widgets.IntSlider(value=3, min=1, max=10, description="Thoroughness:", style={"description_width": "160px"}, layout=widgets.Layout(width="60%"))
w_gypsum_timeout = widgets.IntText(value=15, description="Timeout (min):", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))
w_min_ph = widgets.FloatText(value=6.4, description="Min pH:", style={"description_width": "160px"}, layout=widgets.Layout(width="250px"))
w_max_ph = widgets.FloatText(value=8.4, description="Max pH:", style={"description_width": "160px"}, layout=widgets.Layout(width="250px"))
w_pka_precision = widgets.FloatText(value=1.0, description="pKa precision:", style={"description_width": "160px"}, layout=widgets.Layout(width="250px"))
w_protanate = widgets.Checkbox(value=True, description="Enable ligand protonation (Dimorphite-DL)", style={"description_width": "auto"})

tab_gypsum = widgets.VBox([
    widgets.HTML("<h4>Gypsum-DL Settings (SMILES → 3D Conversion)</h4>"),
    w_max_variants, w_gypsum_thoroughness, w_gypsum_timeout,
    widgets.HTML("<hr><b>pH / Ionization</b>"),
    widgets.HBox([w_min_ph, w_max_ph, w_pka_precision]),
    w_protanate,
])

# ══════════════════════════════════════════════════════════════
# TAB 7: Slurm / HPC Settings
# ══════════════════════════════════════════════════════════════

w_slurm_account = widgets.Text(value="iit130", description="Account:", style={"description_width": "160px"}, layout=widgets.Layout(width="400px"))
w_slurm_partition = widgets.Dropdown(
    options=["shared", "compute", "gpu", "gpu-shared", "large-shared", "debug"],
    value="shared",
    description="Partition:",
    style={"description_width": "160px"}
)
w_slurm_nodes = widgets.IntText(value=1, description="Nodes:", style={"description_width": "160px"}, layout=widgets.Layout(width="250px"))
w_slurm_ntasks = widgets.IntText(value=4, description="Tasks/node:", style={"description_width": "160px"}, layout=widgets.Layout(width="250px"))
w_slurm_mem = widgets.Text(value="10G", description="Memory:", style={"description_width": "160px"}, layout=widgets.Layout(width="250px"))
w_slurm_time = widgets.Text(value="24:00:00", description="Wall time:", style={"description_width": "160px"}, layout=widgets.Layout(width="250px"))
w_slurm_jobname = widgets.Text(value="autogrow4", description="Job name:", style={"description_width": "160px"}, layout=widgets.Layout(width="400px"))
w_slurm_email = widgets.Text(value="", description="Email (optional):", placeholder="your@email.edu", style={"description_width": "160px"}, layout=widgets.Layout(width="400px"))
w_num_procs = widgets.IntText(value=-1, description="Num processors:", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))
w_multithread = widgets.Dropdown(
    options=["multithreading", "mpi", "serial"],
    value="multithreading",
    description="Parallel mode:",
    style={"description_width": "160px"}
)

# GPU configuration
w_gpu_count = widgets.IntText(value=1, description="GPUs:", style={"description_width": "160px"}, layout=widgets.Layout(width="250px"))
w_gpu_type = widgets.Dropdown(
    options=["default", "v100", "a100"],
    value="default",
    description="GPU type:",
    style={"description_width": "160px"},
    layout=widgets.Layout(width="250px")
)
_gpu_box = widgets.HBox([w_gpu_count, w_gpu_type],
    layout=widgets.Layout(display="none"))

def _on_partition_change(change):
    is_gpu = "gpu" in str(change["new"])
    _gpu_box.layout.display = "" if is_gpu else "none"
    if is_gpu:
        # GPU partitions on Expanse need the gpu module
        pass

w_slurm_partition.observe(_on_partition_change, names="value")

# Auto-requeue on timeout
w_requeue = widgets.Checkbox(value=True, description="Auto-requeue on timeout (--requeue)")
w_auto_continue = widgets.Checkbox(value=True, description="Auto-continue from last generation on requeue")

tab_slurm = widgets.VBox([
    widgets.HTML("<h4>Slurm / HPC Settings</h4>"),
    widgets.HBox([w_slurm_account, w_slurm_partition]),
    widgets.HBox([w_slurm_nodes, w_slurm_ntasks]),
    widgets.HBox([w_slurm_mem, w_slurm_time]),
    w_slurm_jobname, w_slurm_email,
    widgets.HTML("<hr><b>Parallelization</b>"),
    widgets.HBox([w_num_procs, w_multithread]),
    widgets.HTML("<hr><b>GPU (for gpu/gpu-shared partitions)</b>"),
    _gpu_box,
    widgets.HTML("<hr><b>Job Recovery</b>"),
    w_requeue, w_auto_continue,
])

# ══════════════════════════════════════════════════════════════
# Assemble Tabs
# ══════════════════════════════════════════════════════════════

tabs = widgets.Tab()
tabs.children = [tab_files, tab_evolution, tab_filters, tab_docking, tab_forcefields, tab_gypsum, tab_slurm]
for i, title in enumerate(["Files/Paths", "Evolution", "Filters", "Docking/Scoring", "Force Fields", "Gypsum-DL", "Slurm/HPC"]):
    tabs.set_title(i, title)

display(tabs)

---

## 4. Generate Configuration & Slurm Script

Click **"Generate & Preview"** to build the AutoGrow4 JSON config and Slurm SBATCH script from the GUI settings above.

In [15]:
import json
import os
from IPython.display import display, HTML
import ipywidgets as widgets

preview_output = widgets.Output()
generate_btn = widgets.Button(description="Generate & Preview", button_style="primary", layout=widgets.Layout(width="250px"))

def build_autogrow_config():
    """Build AutoGrow4 JSON config from widget values."""
    config = {
        # Receptor & binding box
        "filename_of_receptor": w_receptor.value,
        "center_x": binding_box["center_x"],
        "center_y": binding_box["center_y"],
        "center_z": binding_box["center_z"],
        "size_x": binding_box["size_x"],
        "size_y": binding_box["size_y"],
        "size_z": binding_box["size_z"],

        # Files
        "source_compound_file": w_source_compounds.value,
        "root_output_folder": w_output_folder.value,

        # Conversion
        "conversion_choice": w_conversion.value,
        "mgltools_directory": w_mgltools_dir.value if w_conversion.value == "MGLToolsConversion" else "",
        "obabel_path": w_obabel_path.value if w_conversion.value == "ObabelConversion" else "",

        # Evolution
        "num_generations": w_num_gens.value,
        "number_of_mutants_first_generation": w_mutants_first.value,
        "number_of_crossovers_first_generation": w_crossovers_first.value,
        "number_elitism_advance_from_previous_gen_first_generation": w_elitism_first.value,
        "number_of_mutants": w_mutants.value,
        "number_of_crossovers": w_crossovers.value,
        "number_elitism_advance_from_previous_gen": w_elitism.value,
        "top_mols_to_seed_next_generation": w_top_seed.value,
        "diversity_mols_to_seed_first_generation": w_diversity_seed.value,
        "diversity_seed_depreciation_per_gen": w_diversity_deprec.value,

        # Selection
        "selector_choice": w_selector.value,
        "prop_mutations_extending": w_prop_mut_extending.value,
        "prop_mutations_deleting": w_prop_mut_deleting.value,
        "max_attempts_to_generate": w_max_attempts.value,
        "tourn_size": w_tourn_size.value,

        # Reaction library
        "rxn_library": w_rxn_library.value,

        # Options
        "start_a_new_run": w_start_new.value,
        "filter_source_compounds": w_filter_source.value,
        "use_docked_source_compounds": w_use_docked.value,
        "redock_elite_from_previous_gen": w_redock_elite.value,
        "generate_plot": w_generate_plot.value,
        "reduce_files_sizes": w_reduce_files.value,

        # Docking
        "dock_choice": w_dock_choice.value,
        "docking_exhaustiveness": w_exhaustiveness.value,
        "docking_num_modes": w_num_modes.value,
        "docking_timeout_limit": w_dock_timeout.value,

        # Scoring
        "scoring_choice": w_scoring.value,
        "rescore_lig_efficiency": w_lig_efficiency.value,

        # Gypsum-DL
        "max_variants_per_compound": w_max_variants.value,
        "gypsum_thoroughness": w_gypsum_thoroughness.value,
        "gypsum_timeout_limit": w_gypsum_timeout.value,
        "min_ph": w_min_ph.value,
        "max_ph": w_max_ph.value,
        "pka_precision": w_pka_precision.value,
        "protanate_step": w_protanate.value,

        # Parallelization
        "number_of_processors": w_num_procs.value,
        "multithread_mode": w_multithread.value,
        "mgl_python": "",
        "prepare_ligand4.py": "",
        "prepare_receptor4.py": "",
        "custom_conversion_script": "",
        "timeout_limit": 120,
        "cache_prerun": False,
    }

    # Handle agricultural reactions library (treated as custom with preset paths)
    if config.get("rxn_library") == "agro_rxns (Agricultural)":
        _agro_dir = os.path.join(AUTOGROW_DIR,
            "autogrow", "operators", "mutation", "smiles_click_chem",
            "reaction_libraries", "agro_rxns")
        config["rxn_library"] = "Custom"
        config["rxn_library_file"] = os.path.join(_agro_dir, "Agro_Rxns_rxn_library.json")
        config["function_group_library"] = os.path.join(_agro_dir, "Agro_Rxns_functional_groups.json")
        config["complementary_mol_directory"] = os.path.join(_agro_dir, "complementary_mol_dir")

    # Filters
    if w_no_filters.value:
        config["No_Filters"] = True
    else:
        filter_map = {
            "LipinskiStrictFilter": w_lipinski_strict.value,
            "LipinskiLenientFilter": w_lipinski_lenient.value,
            "GhoseFilter": w_ghose.value,
            "GhoseModifiedFilter": w_ghose_modified.value,
            "MozziconacciFilter": w_mozziconacci.value,
            "VandeWaterbeemdFilter": w_vandewaterbeemd.value,
            "PAINSFilter": w_pains.value,
            "NIHFilter": w_nih.value,
            "BRENKFilter": w_brenk.value,
            "CustomLipinskiFilter": w_custom_lipinski.value,
        }
        for k, v in filter_map.items():
            if v:
                config[k] = True

    # Remove empty strings, but keep keys AutoGrow4 expects
    required_keys = {"mgl_python", "prepare_ligand4.py", "prepare_receptor4.py", "custom_conversion_script"}
    config = {k: v for k, v in config.items() if v != "" or k in required_keys}

    return config


def build_slurm_script(config, json_path):
    """Build SBATCH script for Slurm submission."""
    is_gpu = "gpu" in w_slurm_partition.value

    lines = ["#!/bin/bash"]
    lines.append(f"#SBATCH --account={w_slurm_account.value}")
    lines.append(f"#SBATCH --partition={w_slurm_partition.value}")
    lines.append(f"#SBATCH --nodes={w_slurm_nodes.value}")
    lines.append(f"#SBATCH --ntasks-per-node={w_slurm_ntasks.value}")
    lines.append(f"#SBATCH --mem={w_slurm_mem.value}")
    lines.append(f"#SBATCH --time={w_slurm_time.value}")
    lines.append(f"#SBATCH --job-name=\"{w_slurm_jobname.value}\"")
    lines.append(f"#SBATCH --output=\"{w_slurm_jobname.value}.%j.%N.out\"")
    lines.append("#SBATCH --export=ALL")

    # GPU resources
    if is_gpu:
        gpu_type = w_gpu_type.value
        gpu_count = w_gpu_count.value
        if gpu_type == "default":
            lines.append(f"#SBATCH --gpus={gpu_count}")
        else:
            lines.append(f"#SBATCH --gpus={gpu_type}:{gpu_count}")

    # Auto-requeue on timeout
    if w_requeue.value:
        lines.append("#SBATCH --requeue")

    if w_slurm_email.value.strip():
        lines.append(f"#SBATCH --mail-user={w_slurm_email.value.strip()}")
        lines.append("#SBATCH --mail-type=ALL")

    lines.append("")
    lines.append("# ── Environment setup ──")
    lines.append("module purge")
    if is_gpu:
        lines.append("module load gpu")
    else:
        lines.append("module load cpu")
    lines.append("")
    lines.append("# Activate conda environment with all AutoGrow4 dependencies")
    lines.append("source ~/miniconda3/etc/profile.d/conda.sh")
    lines.append("conda activate ag4_full")
    lines.append("")
    lines.append(f"cd {AUTOGROW_DIR}")
    lines.append("")

    # Auto-continue: check if this is a requeued job and adjust config
    if w_auto_continue.value:
        lines.append("# Auto-continue: if output dir has completed generations, continue from there")
        lines.append(f"CONFIG_FILE=\"{json_path}\"")
        lines.append(f"OUTPUT_DIR=\"{config.get('output_folder', '')}\"")
        lines.append("if [ -d \"$OUTPUT_DIR\" ]; then")
        lines.append("    LAST_RUN=$(ls -d \"$OUTPUT_DIR\"/Run_* 2>/dev/null | sort -V | tail -1)")
        lines.append("    if [ -n \"$LAST_RUN\" ]; then")
        lines.append("        LAST_GEN=$(ls -d \"$LAST_RUN\"/generation_* 2>/dev/null | sort -V | tail -1)")
        lines.append("        if [ -n \"$LAST_GEN\" ]; then")
        lines.append("            GEN_NUM=$(basename \"$LAST_GEN\" | sed 's/generation_//')")
        lines.append("            echo \"Resuming from generation $GEN_NUM in $LAST_RUN\"")
        lines.append("            # Create a continue config that picks up where we left off")
        lines.append("            python3 -c \"")
        lines.append("import json")
        lines.append("with open('$CONFIG_FILE') as f: cfg = json.load(f)")
        lines.append("cfg['start_a_new_run'] = False")
        lines.append("with open('$CONFIG_FILE', 'w') as f: json.dump(cfg, f, indent=4)")
        lines.append("print('Config updated: start_a_new_run = False')")
        lines.append("\"")
        lines.append("        fi")
        lines.append("    fi")
        lines.append("fi")
        lines.append("")

    # Cache prerun first (prevents race conditions)
    lines.append("# Step 1: Cache prerun (single processor, prevents race conditions)")
    if w_multithread.value == "mpi":
        lines.append(f"python run_autogrow.py -c")
        lines.append("")
        lines.append("# Step 2: Run AutoGrow4 with MPI")
        ntasks_total = w_slurm_nodes.value * w_slurm_ntasks.value
        lines.append(f"mpirun -n {ntasks_total} python -m mpi4py run_autogrow.py -j {json_path}")
    else:
        lines.append(f"python run_autogrow.py -c")
        lines.append("")
        lines.append("# Step 2: Run AutoGrow4 with multiprocessing")
        lines.append(f"python run_autogrow.py -j {json_path}")

    lines.append("")
    lines.append("PYTHON_EXIT=$?")
    lines.append("if [ $PYTHON_EXIT -ne 0 ]; then")
    lines.append("    echo 'AutoGrow4 FAILED with exit code $PYTHON_EXIT'")
    lines.append("    exit $PYTHON_EXIT")
    lines.append("fi")
    lines.append("echo 'AutoGrow4 run completed successfully.'")


    return "\n".join(lines)


def on_generate(btn):
    preview_output.clear_output()
    with preview_output:
        config = build_autogrow_config()
        json_path = os.path.join(PROJECT_DIR, "autogrow_config.json")
        slurm_path = os.path.join(PROJECT_DIR, "autogrow_slurm.sh")

        # Write custom filter config if enabled
        if w_custom_lipinski.value:
            custom_filter_cfg = {
                "mw_enabled": w_mw_enable.value,
                "mw_min": w_mw_min.value,
                "mw_max": w_mw_max.value,
                "logp_enabled": w_logp_enable.value,
                "logp_min": w_logp_min.value,
                "logp_max": w_logp_max.value,
                "h_donors_enabled": w_hdonors_enable.value,
                "h_donors_min": w_hdonors_min.value,
                "h_donors_max": w_hdonors_max.value,
                "h_acceptors_enabled": w_hacceptors_enable.value,
                "h_acceptors_min": w_hacceptors_min.value,
                "h_acceptors_max": w_hacceptors_max.value,
                "rot_bonds_enabled": w_rotbonds_enable.value,
                "rot_bonds_min": w_rotbonds_min.value,
                "rot_bonds_max": w_rotbonds_max.value,
                "tpsa_enabled": w_tpsa_enable.value,
                "tpsa_min": w_tpsa_min.value,
                "tpsa_max": w_tpsa_max.value,
                "aromatic_rings_enabled": w_aromrings_enable.value,
                "aromatic_rings_min": w_aromrings_min.value,
                "aromatic_rings_max": w_aromrings_max.value,
            }
            custom_cfg_path = os.path.join(PROJECT_DIR, "custom_filter_config.json")
            with open(custom_cfg_path, "w") as f:
                json.dump(custom_filter_cfg, f, indent=4)
            parts = []
            if w_mw_enable.value: parts.append(f"MW {w_mw_min.value}-{w_mw_max.value}")
            if w_logp_enable.value: parts.append(f"LogP {w_logp_min.value}-{w_logp_max.value}")
            if w_hdonors_enable.value: parts.append(f"H-donors {w_hdonors_min.value}-{w_hdonors_max.value}")
            if w_hacceptors_enable.value: parts.append(f"H-acceptors {w_hacceptors_min.value}-{w_hacceptors_max.value}")
            if w_rotbonds_enable.value: parts.append(f"Rot-bonds {w_rotbonds_min.value}-{w_rotbonds_max.value}")
            if w_tpsa_enable.value: parts.append(f"TPSA {w_tpsa_min.value}-{w_tpsa_max.value}")
            if w_aromrings_enable.value: parts.append(f"AromRings {w_aromrings_min.value}-{w_aromrings_max.value}")
            display(HTML(f"<p>Custom filter config saved: {', '.join(parts) if parts else 'All parameters disabled'}</p>"))

        # Save JSON config
        with open(json_path, "w") as f:
            json.dump(config, f, indent=4)

        # Build and save Slurm script
        slurm_script = build_slurm_script(config, json_path)
        with open(slurm_path, "w") as f:
            f.write(slurm_script)

        display(HTML(f"<h4>AutoGrow4 Config: <code>{json_path}</code></h4>"))
        print(json.dumps(config, indent=2))

        display(HTML(f"<hr><h4>Slurm Script: <code>{slurm_path}</code></h4>"))
        print(slurm_script)

        display(HTML("<hr><p style='color:green'><b>Files saved! Ready to submit.</b></p>"))

generate_btn.on_click(on_generate)

display(generate_btn)
display(preview_output)

Button(button_style='primary', description='Generate & Preview', layout=Layout(width='250px'), style=ButtonSty…

Output()

---

## 5. Submit Job to Expanse

Click **"Submit Job"** to launch the AutoGrow4 run on SDSC Expanse via Slurm.

In [16]:
import subprocess
import ipywidgets as widgets
from IPython.display import display, HTML

submit_output = widgets.Output()
submit_btn = widgets.Button(description="Submit Job", button_style="success", layout=widgets.Layout(width="200px"))

slurm_job_id = [None]  # Track for monitoring

def on_submit(btn):
    submit_output.clear_output()
    with submit_output:
        slurm_path = os.path.join(PROJECT_DIR, "autogrow_slurm.sh")
        json_path = os.path.join(PROJECT_DIR, "autogrow_config.json")

        if not os.path.exists(slurm_path):
            display(HTML("<span style='color:red'>No Slurm script found. Click 'Generate & Preview' first.</span>"))
            return

        if not os.path.exists(json_path):
            display(HTML("<span style='color:red'>No config JSON found. Click 'Generate & Preview' first.</span>"))
            return

        print("Submitting job to Slurm...")
        try:
            result = subprocess.run(
                ["sbatch", slurm_path],
                capture_output=True, text=True, cwd=PROJECT_DIR
            )
            if result.returncode == 0:
                output = result.stdout.strip()
                print(f"Slurm response: {output}")
                # Extract job ID
                parts = output.split()
                if len(parts) >= 4:
                    slurm_job_id[0] = parts[-1]
                    display(HTML(f"<p style='color:green'><b>Job submitted successfully! Job ID: {slurm_job_id[0]}</b></p>"))
                else:
                    display(HTML(f"<p style='color:green'><b>Job submitted!</b> {output}</p>"))
            else:
                print(f"STDERR: {result.stderr}")
                display(HTML(f"<span style='color:red'>Submission failed: {result.stderr}</span>"))
        except FileNotFoundError:
            display(HTML("<span style='color:red'>sbatch not found. Are you on a Slurm-enabled node?</span>"))
        except Exception as e:
            display(HTML(f"<span style='color:red'>Error: {e}</span>"))

submit_btn.on_click(on_submit)

display(submit_btn)
display(submit_output)

Button(button_style='success', description='Submit Job', layout=Layout(width='200px'), style=ButtonStyle())

Output()

---

## 6. Monitor Job & Retrieve Results

Use the cells below to check job status, stream logs, and view results.

In [17]:
import subprocess
import glob
import ipywidgets as widgets
from IPython.display import display, HTML

monitor_output = widgets.Output()
check_btn = widgets.Button(description="Check Job Status", button_style="info", layout=widgets.Layout(width="200px"))
logs_btn = widgets.Button(description="View Latest Logs", button_style="warning", layout=widgets.Layout(width="200px"))
cancel_btn = widgets.Button(description="Cancel Job", button_style="danger", layout=widgets.Layout(width="200px"))

def on_check(btn):
    monitor_output.clear_output()
    with monitor_output:
        try:
            # Show queue status
            result = subprocess.run(["squeue", "-u", os.environ.get("USER", "$USER"), "--format=%.18i %.9P %.30j %.8u %.8T %.10M %.9l %.6D %R"],
                                   capture_output=True, text=True)
            if result.stdout.strip():
                print("=== Your Active Jobs ===")
                print(result.stdout)
            else:
                print("No active jobs found in the queue.")

            # Check specific job if we have an ID
            if slurm_job_id[0]:
                result2 = subprocess.run(["sacct", "-j", slurm_job_id[0], "--format=JobID,State,ExitCode,Elapsed,MaxRSS"],
                                        capture_output=True, text=True)
                if result2.stdout.strip():
                    print(f"\n=== Job {slurm_job_id[0]} Details ===")
                    print(result2.stdout)
        except FileNotFoundError:
            print("Slurm commands not available. Are you on a login/compute node?")

def on_logs(btn):
    monitor_output.clear_output()
    with monitor_output:
        # Find the most recent .out file
        pattern = os.path.join(PROJECT_DIR, f"{w_slurm_jobname.value}.*.out")
        files = sorted(glob.glob(pattern), key=os.path.getmtime, reverse=True)

        if files:
            latest = files[0]
            print(f"=== Latest log: {latest} ===")
            print(f"(showing last 50 lines)\n")
            with open(latest) as f:
                lines = f.readlines()
                for line in lines[-50:]:
                    print(line, end="")
        else:
            print("No log files found yet. The job may not have started.")

        # Also check AutoGrow output directory
        output = w_output_folder.value
        if os.path.exists(output):
            gen_dirs = sorted(glob.glob(os.path.join(output, "generation_*")))
            if gen_dirs:
                print(f"\n=== AutoGrow4 Progress ===")
                print(f"Completed generations: {len(gen_dirs)}")
                for gd in gen_dirs[-3:]:
                    smi_files = glob.glob(os.path.join(gd, "*.smi"))
                    print(f"  {os.path.basename(gd)}: {len(smi_files)} .smi files")

def on_cancel(btn):
    monitor_output.clear_output()
    with monitor_output:
        if slurm_job_id[0]:
            result = subprocess.run(["scancel", slurm_job_id[0]], capture_output=True, text=True)
            print(f"Cancelled job {slurm_job_id[0]}")
            if result.stderr:
                print(f"Warning: {result.stderr}")
        else:
            print("No job ID tracked. Use: scancel <job_id>")

check_btn.on_click(on_check)
logs_btn.on_click(on_logs)
cancel_btn.on_click(on_cancel)

display(widgets.HBox([check_btn, logs_btn, cancel_btn]))
display(monitor_output)

Output()

---

## 7. Analyze Results

After the AutoGrow4 run completes, use the cells below to inspect the results.

In [ ]:
import glob
import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ══════════════════════════════════════════════════════════════
# Standalone Results Viewer - works without running the GUI cell
# ══════════════════════════════════════════════════════════════

PROJECT_DIR = os.path.expanduser("~/final_project")

def _find_output_dirs():
    """Find all directories containing AutoGrow4 Run_* output folders."""
    output_dirs = []
    # Check common output locations
    for candidate in glob.glob(os.path.join(PROJECT_DIR, "*")):
        if os.path.isdir(candidate):
            runs = glob.glob(os.path.join(candidate, "Run_*"))
            if runs:
                output_dirs.append(candidate)
    # Also check if any Run_* folders are directly in PROJECT_DIR
    if glob.glob(os.path.join(PROJECT_DIR, "Run_*")):
        output_dirs.append(PROJECT_DIR)
    return sorted(set(output_dirs))

def _find_runs(output_dir):
    """Find all Run_* folders in an output directory."""
    runs = sorted(glob.glob(os.path.join(output_dir, "Run_*")))
    return runs

def _find_generations(run_dir):
    """Find all generation folders with ranked .smi files."""
    ranked = sorted(glob.glob(os.path.join(run_dir, "generation_*", "*ranked*.smi")))
    return ranked

def load_ranked_smi(filepath):
    """Load a ranked .smi file into a DataFrame."""
    rows = []
    with open(filepath) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 2:
                row = {"SMILES": parts[0], "Name": parts[1]}
                if len(parts) >= 3:
                    try: row["Docking_Score"] = float(parts[2])
                    except ValueError: pass
                if len(parts) >= 4:
                    try: row["Diversity_Score"] = float(parts[3])
                    except ValueError: pass
                rows.append(row)
    return pd.DataFrame(rows)

# --- Build UI ---
_output_dirs = _find_output_dirs()
_output_labels = {d: os.path.basename(d) for d in _output_dirs}

w_results_output_dir = widgets.Dropdown(
    options=[(os.path.basename(d), d) for d in _output_dirs] if _output_dirs else [("No output directories found", "")],
    description="Output Dir:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="500px")
)
w_results_run = widgets.Dropdown(
    options=[],
    description="Run:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="500px")
)
w_results_gen = widgets.Dropdown(
    options=[],
    description="Generation:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="500px")
)
_refresh_results_btn = widgets.Button(description="Refresh", button_style="info", layout=widgets.Layout(width="100px"))
_show_results_btn = widgets.Button(description="Show Results", button_style="success", layout=widgets.Layout(width="150px"))
results_output = widgets.Output()

def _update_runs(change=None):
    """Update Run dropdown when output dir changes."""
    output_dir = w_results_output_dir.value
    if output_dir:
        runs = _find_runs(output_dir)
        w_results_run.options = [(os.path.basename(r), r) for r in runs] if runs else [("No runs found", "")]
        if runs:
            _update_gens()

def _update_gens(change=None):
    """Update Generation dropdown when Run changes."""
    run_dir = w_results_run.value
    if run_dir:
        ranked = _find_generations(run_dir)
        gen_labels = []
        for r in ranked:
            gen_name = os.path.basename(os.path.dirname(r))
            size = os.path.getsize(r)
            gen_labels.append((f"{gen_name} ({size} bytes)", r))
        w_results_gen.options = gen_labels if gen_labels else [("No ranked files found", "")]

def _refresh_all(b=None):
    """Refresh all dropdowns."""
    dirs = _find_output_dirs()
    w_results_output_dir.options = [(os.path.basename(d), d) for d in dirs] if dirs else [("No output directories found", "")]
    if dirs:
        _update_runs()

def _show_results(b):
    """Display results for selected generation."""
    results_output.clear_output()
    with results_output:
        ranked_file = w_results_gen.value
        if not ranked_file or not os.path.exists(ranked_file):
            print("No ranked file selected or file not found.")
            return

        gen_name = os.path.basename(os.path.dirname(ranked_file))
        run_name = os.path.basename(os.path.dirname(os.path.dirname(ranked_file)))
        display(HTML(f"<h4>Results: {run_name} / {gen_name}</h4>"))

        df = load_ranked_smi(ranked_file)
        if df.empty:
            print("Ranked file is empty (0 bytes). All dockings may have failed.")
            return

        display(HTML(f"<p><b>{len(df)} ranked molecules</b></p>"))
        if "Docking_Score" in df.columns:
            display(HTML(f"<p>Best docking score: {df['Docking_Score'].min():.2f} kcal/mol</p>"))
            display(HTML(f"<p>Average docking score: {df['Docking_Score'].mean():.2f} kcal/mol</p>"))

        display(HTML("<p><b>Top 20 molecules:</b></p>"))
        display(df.head(20))

        # Also show all generations summary for this run
        run_dir = w_results_run.value
        all_ranked = _find_generations(run_dir)
        if len(all_ranked) > 1:
            display(HTML("<hr><h4>All Generations Summary</h4>"))
            summary_rows = []
            for rf in all_ranked:
                gname = os.path.basename(os.path.dirname(rf))
                gdf = load_ranked_smi(rf)
                if not gdf.empty and "Docking_Score" in gdf.columns:
                    summary_rows.append({
                        "Generation": gname,
                        "Molecules": len(gdf),
                        "Best_Score": gdf["Docking_Score"].min(),
                        "Avg_Score": gdf["Docking_Score"].mean(),
                        "Worst_Score": gdf["Docking_Score"].max()
                    })
                else:
                    summary_rows.append({"Generation": gname, "Molecules": len(gdf),
                        "Best_Score": None, "Avg_Score": None, "Worst_Score": None})
            if summary_rows:
                display(pd.DataFrame(summary_rows))

w_results_output_dir.observe(_update_runs, names="value")
w_results_run.observe(_update_gens, names="value")
_refresh_results_btn.on_click(_refresh_all)
_show_results_btn.on_click(_show_results)

# Initialize
_update_runs()

display(widgets.VBox([
    widgets.HTML("<h3>AutoGrow4 Results Viewer</h3>"),
    widgets.HTML("<p>Select an output directory, run, and generation to view results. Click Refresh to rescan.</p>"),
    widgets.HBox([w_results_output_dir, _refresh_results_btn]),
    w_results_run,
    w_results_gen,
    _show_results_btn,
    results_output
]))


In [ ]:
# ── Fitness Progression Plot (standalone) ──
import matplotlib.pyplot as plt
import numpy as np

plot_output = widgets.Output()
_plot_btn = widgets.Button(description="Plot Fitness", button_style="primary", layout=widgets.Layout(width="150px"))

def _plot_fitness(b):
    plot_output.clear_output()
    with plot_output:
        run_dir = w_results_run.value
        if not run_dir:
            print("Select a Run first in the Results Viewer above.")
            return

        ranked_files = sorted(glob.glob(os.path.join(run_dir, "generation_*", "*ranked*.smi")))
        if not ranked_files:
            print("No ranked files found in this run.")
            return

        gen_numbers = []
        best_scores = []
        avg_scores = []

        for rf in ranked_files:
            gen_dir = os.path.basename(os.path.dirname(rf))
            gen_num = int(gen_dir.replace("generation_", ""))
            scores = []
            with open(rf) as f:
                for line in f:
                    parts = line.strip().split("\t")
                    if len(parts) >= 3:
                        try: scores.append(float(parts[2]))
                        except ValueError: pass
            if scores:
                gen_numbers.append(gen_num)
                best_scores.append(min(scores))
                avg_scores.append(np.mean(scores))

        if gen_numbers:
            fig, ax = plt.subplots(figsize=(10, 5))
            ax.plot(gen_numbers, best_scores, "o-", label="Best Score", color="#e74c3c", linewidth=2)
            ax.plot(gen_numbers, avg_scores, "s--", label="Average Score", color="#3498db", linewidth=1.5)
            ax.set_xlabel("Generation", fontsize=12)
            ax.set_ylabel("Docking Score (kcal/mol)", fontsize=12)
            ax.set_title("AutoGrow4 Fitness Progression", fontsize=14)
            ax.legend()
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
        else:
            print("No valid scores found in ranked files.")

_plot_btn.on_click(_plot_fitness)
display(widgets.VBox([_plot_btn, plot_output]))


In [ ]:
# ── Docking Visualization (standalone) ──
import nglview as nv

viz_output = widgets.Output()

# Ligand selector dropdown
w_ligand_select = widgets.Dropdown(
    options=[],
    description="Ligand:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="600px")
)
w_receptor_path = widgets.Text(
    value=os.path.expanduser("~/final_project/receptors/2ZSH.pdbqt"),
    description="Receptor:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="600px")
)
_load_ligands_btn = widgets.Button(description="Load Ligands", button_style="info", layout=widgets.Layout(width="150px"))
_show_dock_btn = widgets.Button(description="Show Docking", button_style="success", layout=widgets.Layout(width="150px"))

def _load_ligand_list(b):
    """Find all docked ligand files in the selected run's latest generation."""
    viz_output.clear_output()
    with viz_output:
        run_dir = w_results_run.value
        if not run_dir:
            print("Select a Run first in the Results Viewer above.")
            return

        gen_dirs = sorted(glob.glob(os.path.join(run_dir, "generation_*")))
        if not gen_dirs:
            print("No generation directories found.")
            return

        latest_gen = gen_dirs[-1]

        # Check if PDBs directory has compressed files
        pdbs_dir = os.path.join(latest_gen, "PDBs")
        compressed = os.path.join(pdbs_dir, "compressed_PDBS.txt.gz")
        if os.path.exists(compressed) and not glob.glob(os.path.join(pdbs_dir, "*.pdb")):
            print(f"PDB files are compressed in {os.path.basename(latest_gen)}/PDBs/compressed_PDBS.txt.gz")
            print("Decompressing...")
            import gzip, shutil
            # AutoGrow4 compresses all PDB/PDBQT files into this archive
            # We need to extract them first
            try:
                with gzip.open(compressed, "rt") as gz:
                    content = gz.read()
                # The compressed file contains all PDB file contents concatenated
                # with headers like ">>>>filename.pdb"
                current_file = None
                current_lines = []
                for line in content.split("\n"):
                    if line.startswith(">>>>"):
                        if current_file and current_lines:
                            with open(os.path.join(pdbs_dir, current_file), "w") as wf:
                                wf.write("\n".join(current_lines))
                        current_file = line[4:].strip()
                        current_lines = []
                    else:
                        current_lines.append(line)
                if current_file and current_lines:
                    with open(os.path.join(pdbs_dir, current_file), "w") as wf:
                        wf.write("\n".join(current_lines))
                print("Decompression complete.")
            except Exception as e:
                print(f"Could not decompress: {e}")
                print("Try running: cd {pdbs_dir} && gunzip -k compressed_PDBS.txt.gz")
                return

        # Find ligand files (prefer .pdb over .pdbqt)
        pdb_files = sorted(glob.glob(os.path.join(pdbs_dir, "*.pdb")))
        pdbqt_files = sorted(glob.glob(os.path.join(pdbs_dir, "*.pdbqt")))
        # Also check .vina files (docked poses)
        vina_files = sorted(glob.glob(os.path.join(pdbs_dir, "*.vina")))

        all_ligands = pdb_files + pdbqt_files + vina_files
        if all_ligands:
            w_ligand_select.options = [(os.path.basename(f), f) for f in all_ligands]
            print(f"Found {len(pdb_files)} PDB, {len(pdbqt_files)} PDBQT, {len(vina_files)} VINA files in {os.path.basename(latest_gen)}")
        else:
            print(f"No ligand files found in {os.path.basename(latest_gen)}/PDBs/")
            print("Files may have been cleaned up. Check if reduce_files was enabled.")

def _show_docking(b):
    """Show 3D visualization of selected ligand with receptor."""
    viz_output.clear_output()
    with viz_output:
        ligand_file = w_ligand_select.value
        receptor_file = w_receptor_path.value

        if not ligand_file:
            print("Select a ligand first (click Load Ligands).")
            return

        if not os.path.exists(ligand_file):
            print(f"Ligand file not found: {ligand_file}")
            return

        print(f"Loading: {os.path.basename(ligand_file)}")

        try:
            if os.path.exists(receptor_file):
                view = nv.show_file(receptor_file)
                view.clear_representations()
                view.add_cartoon(selection="protein", color="sstruc", opacity=0.7)
                view.add_component(ligand_file)
                view[1].add_licorice(color="element")
                view[1].add_surface(opacity=0.3, color="element")
            else:
                print(f"Receptor not found at {receptor_file}, showing ligand only.")
                view = nv.show_file(ligand_file)
                view.add_licorice(color="element")

            display(view)
        except Exception as e:
            print(f"Visualization error: {e}")
            print("nglview may not support this file format. Try a .pdb file.")

_load_ligands_btn.on_click(_load_ligand_list)
_show_dock_btn.on_click(_show_docking)

display(widgets.VBox([
    widgets.HTML("<h3>Docking Visualization</h3>"),
    w_receptor_path,
    widgets.HBox([_load_ligands_btn, _show_dock_btn]),
    w_ligand_select,
    viz_output
]))


### 7.3 Advanced Trial Analysis (Hybrid Metrics)

Comprehensive analysis of AutoGrow4 hybrid trials with 6 key metrics:
1. **MW Distribution Shift** — Did hybridization change the molecular weight profile?
2. **Score vs MW (Pareto Frontier)** — Which molecules have the best scores at the lowest weights?
3. **Reaction Frequency** — Which reactions (including custom agro reactions) drove the top scorers?
4. **Fragment Survival Rate** — How many small fragments survived through evolution?
5. **Physicochemical Drift** — Are LogP and TPSA trending toward or away from phloem mobility?
6. **Score Progression** — Best and average Vina scores across generations

In [ ]:
# ── Advanced Trial Analysis (5 Metrics) ──
import glob, os, re
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Crippen
    from rdkit import RDLogger
    RDLogger.DisableLog("rdApp.*")
    HAS_RDKIT = True
except ImportError:
    HAS_RDKIT = False

PROJECT_DIR = os.path.expanduser("~/final_project")
DARK_COLORS = ["#1b4332", "#023e8a", "#6a040f", "#7b2d8e", "#b5651d",
               "#1a5276", "#7d3c98", "#196f3d", "#922b21", "#2c3e50"]

def _load_all_gens(run_dir):
    ranked = sorted(glob.glob(os.path.join(run_dir, "generation_*", "*ranked*.smi")))
    gens = {}
    for rf in ranked:
        gname = os.path.basename(os.path.dirname(rf))
        m = re.search(r"generation_(\d+)", gname)
        if not m:
            continue
        gnum = int(m.group(1))
        rows = []
        with open(rf) as f:
            for line in f:
                parts = line.strip().split("\t")
                if len(parts) >= 2:
                    row = {"SMILES": parts[0], "Name": parts[1]}
                    if len(parts) >= 3:
                        try: row["Docking_Score"] = float(parts[2])
                        except: pass
                    if len(parts) >= 4:
                        try: row["Diversity_Score"] = float(parts[3])
                        except: pass
                    rows.append(row)
        gens[gnum] = pd.DataFrame(rows)
    return gens

def _compute_properties(df):
    if not HAS_RDKIT or df.empty:
        return df
    mws, logps, tpsas, ha_counts = [], [], [], []
    for smi in df["SMILES"]:
        mol = Chem.MolFromSmiles(smi) if smi else None
        if mol:
            mws.append(Descriptors.ExactMolWt(mol))
            logps.append(Crippen.MolLogP(mol))
            tpsas.append(Descriptors.TPSA(mol))
            ha_counts.append(mol.GetNumHeavyAtoms())
        else:
            mws.append(np.nan); logps.append(np.nan)
            tpsas.append(np.nan); ha_counts.append(np.nan)
    df = df.copy()
    df["MW"] = mws; df["LogP"] = logps
    df["TPSA"] = tpsas; df["HeavyAtoms"] = ha_counts
    if "Docking_Score" in df.columns:
        df["LE"] = df["Docking_Score"] / df["HeavyAtoms"]
    return df

def _count_reactions(names, rxn_json_path=None):
    rxn_counts = {}
    for name in names:
        rxns = re.findall(r"__([A-Za-z_]+?)__", str(name))
        for r in rxns:
            rxn_counts[r] = rxn_counts.get(r, 0) + 1
    return rxn_counts

analysis_output = widgets.Output()
analysis_btn = widgets.Button(description="Run Advanced Analysis", button_style="warning",
                              layout=widgets.Layout(width="250px"))
w_analysis_run = widgets.Dropdown(options=[], description="Run:", style={"description_width": "80px"},
                                  layout=widgets.Layout(width="500px"))
w_analysis_topn = widgets.IntText(value=100, description="Top N:", style={"description_width": "80px"},
                                  layout=widgets.Layout(width="200px"))
w_analysis_mw_threshold = widgets.FloatText(value=100.0, description="Fragment MW cap:",
                                            style={"description_width": "110px"},
                                            layout=widgets.Layout(width="250px"))
_analysis_refresh = widgets.Button(description="Refresh", button_style="info",
                                   layout=widgets.Layout(width="80px"))

def _refresh_analysis_runs(b=None):
    all_runs = []
    for candidate in glob.glob(os.path.join(PROJECT_DIR, "*")):
        if os.path.isdir(candidate):
            runs = sorted(glob.glob(os.path.join(candidate, "Run_*")))
            for r in runs:
                ranked = glob.glob(os.path.join(r, "generation_*", "*ranked*.smi"))
                if ranked:
                    label = os.path.basename(os.path.dirname(r)) + "/" + os.path.basename(r)
                    all_runs.append((label, r))
    w_analysis_run.options = all_runs if all_runs else [("No runs found", "")]
_refresh_analysis_runs()

def run_analysis(b):
    analysis_output.clear_output()
    with analysis_output:
        run_dir = w_analysis_run.value
        if not run_dir or not os.path.isdir(run_dir):
            print("Select a valid run directory.")
            return
        top_n = w_analysis_topn.value
        frag_mw_cap = w_analysis_mw_threshold.value
        run_name = os.path.basename(run_dir)
        display(HTML(f"<h3>Advanced Analysis: {run_name}</h3>"))

        gens = _load_all_gens(run_dir)
        if not gens:
            print("No generation data found.")
            return
        gen_nums = sorted(gens.keys())
        last_gen = gen_nums[-1]
        first_gen = gen_nums[0]

        display(HTML(f"<p>Generations found: {first_gen} to {last_gen} | "
                     f"Computing properties for {sum(len(g) for g in gens.values())} molecules...</p>"))
        for gn in gen_nums:
            gens[gn] = _compute_properties(gens[gn])

        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(f"Trial Analysis: {run_name}", fontsize=14, fontweight="bold", color="black")
        plt.rcParams.update({"text.color": "black", "axes.labelcolor": "black",
                             "xtick.color": "black", "ytick.color": "black"})

        # ── 1. Hybridization Success Ratio (MW Distribution Shift) ──
        ax = axes[0, 0]
        if "MW" in gens[first_gen].columns and "MW" in gens[last_gen].columns:
            ax.hist(gens[first_gen]["MW"].dropna(), bins=30, alpha=0.6, color=DARK_COLORS[0],
                    label=f"Gen {first_gen}", edgecolor="black")
            ax.hist(gens[last_gen]["MW"].dropna(), bins=30, alpha=0.6, color=DARK_COLORS[1],
                    label=f"Gen {last_gen}", edgecolor="black")
            ax.set_xlabel("Molecular Weight (Da)", color="black")
            ax.set_ylabel("Count", color="black")
            ax.set_title("1. MW Distribution Shift", color="black", fontweight="bold")
            ax.legend(facecolor="white", edgecolor="black")
            avg0 = gens[first_gen]["MW"].mean()
            avgL = gens[last_gen]["MW"].mean()
            ax.axvline(avg0, color=DARK_COLORS[0], linestyle="--", linewidth=1.5)
            ax.axvline(avgL, color=DARK_COLORS[1], linestyle="--", linewidth=1.5)
            ax.text(0.02, 0.95, f"Gen {first_gen} avg: {avg0:.0f} Da\nGen {last_gen} avg: {avgL:.0f} Da",
                    transform=ax.transAxes, va="top", fontsize=9, color="black",
                    bbox=dict(boxstyle="round", facecolor="white", edgecolor="gray"))
        ax.tick_params(colors="black")

        # ── 2. LE vs Raw Vina Score (Pareto Frontier) ──
        ax = axes[0, 1]
        df_last = gens[last_gen]
        if "MW" in df_last.columns and "Docking_Score" in df_last.columns:
            sc = ax.scatter(df_last["MW"], df_last["Docking_Score"], c=DARK_COLORS[2],
                           s=15, alpha=0.5, edgecolors="none")
            top_df = df_last.nsmallest(top_n, "Docking_Score")
            ax.scatter(top_df["MW"], top_df["Docking_Score"], c=DARK_COLORS[3],
                      s=30, alpha=0.8, edgecolors="black", linewidths=0.5, label=f"Top {top_n}")
            ax.set_xlabel("Molecular Weight (Da)", color="black")
            ax.set_ylabel("Vina Score (kcal/mol)", color="black")
            ax.set_title("2. Score vs MW (Pareto Frontier)", color="black", fontweight="bold")
            ax.legend(facecolor="white", edgecolor="black")
        ax.tick_params(colors="black")

        # ── 3. Reaction Frequency in Top Scorers ──
        ax = axes[0, 2]
        if not df_last.empty:
            top_names = df_last.nsmallest(min(250, len(df_last)), "Docking_Score")["Name"].tolist() if "Docking_Score" in df_last.columns else df_last["Name"].tolist()[:250]
            rxn_counts = _count_reactions(top_names)
            if rxn_counts:
                sorted_rxns = sorted(rxn_counts.items(), key=lambda x: -x[1])[:20]
                rnames = [r[0][:25] for r in sorted_rxns]
                rcounts = [r[1] for r in sorted_rxns]
                bars = ax.barh(range(len(rnames)), rcounts, color=DARK_COLORS[4], edgecolor="black")
                ax.set_yticks(range(len(rnames)))
                ax.set_yticklabels(rnames, fontsize=7, color="black")
                ax.set_xlabel("Frequency in Top 250", color="black")
                ax.set_title("3. Reaction Frequency", color="black", fontweight="bold")
                ax.invert_yaxis()
            else:
                ax.text(0.5, 0.5, "No reaction patterns\nfound in names", ha="center",
                       va="center", transform=ax.transAxes, color="black")
                ax.set_title("3. Reaction Frequency", color="black", fontweight="bold")
        ax.tick_params(colors="black")

        # ── 4. Fragment Survival Rate ──
        ax = axes[1, 0]
        frag_counts = []
        for gn in gen_nums:
            gdf = gens[gn]
            if "MW" in gdf.columns:
                n_frags = (gdf["MW"] <= frag_mw_cap).sum()
                n_total = len(gdf)
                frag_counts.append({"Gen": gn, "Fragments": n_frags, "Total": n_total,
                                    "Pct": 100.0 * n_frags / n_total if n_total > 0 else 0})
        if frag_counts:
            fdf = pd.DataFrame(frag_counts)
            ax.bar(fdf["Gen"], fdf["Pct"], color=DARK_COLORS[5], edgecolor="black")
            ax.set_xlabel("Generation", color="black")
            ax.set_ylabel(f"% Molecules <= {frag_mw_cap} Da", color="black")
            ax.set_title(f"4. Fragment Survival (MW <= {frag_mw_cap} Da)", color="black", fontweight="bold")
            for _, row in fdf.iterrows():
                ax.text(row["Gen"], row["Pct"] + 0.5, f"{row['Fragments']}/{row['Total']}",
                       ha="center", fontsize=7, color="black")
        ax.tick_params(colors="black")

        # ── 5. Physicochemical Drift (LogP & TPSA) ──
        ax = axes[1, 1]
        ax2 = ax.twinx()
        logp_avgs, tpsa_avgs = [], []
        for gn in gen_nums:
            gdf = gens[gn]
            if "Docking_Score" in gdf.columns:
                top = gdf.nsmallest(min(top_n, len(gdf)), "Docking_Score")
            else:
                top = gdf.head(min(top_n, len(gdf)))
            logp_avgs.append(top["LogP"].mean() if "LogP" in top.columns else np.nan)
            tpsa_avgs.append(top["TPSA"].mean() if "TPSA" in top.columns else np.nan)
        l1, = ax.plot(gen_nums, logp_avgs, "o-", color=DARK_COLORS[6], linewidth=2, markersize=6, label="Avg LogP")
        l2, = ax2.plot(gen_nums, tpsa_avgs, "s--", color=DARK_COLORS[7], linewidth=2, markersize=6, label="Avg TPSA")
        ax.set_xlabel("Generation", color="black")
        ax.set_ylabel("Average LogP (top {})".format(top_n), color=DARK_COLORS[6])
        ax2.set_ylabel("Average TPSA (top {})".format(top_n), color=DARK_COLORS[7])
        ax.set_title("5. Physicochemical Drift", color="black", fontweight="bold")
        ax.axhline(3.0, color="gray", linestyle=":", linewidth=1, label="LogP=3 (greasy)")
        lines = [l1, l2]
        ax.legend(lines, [l.get_label() for l in lines], loc="upper left",
                 facecolor="white", edgecolor="black")
        ax.tick_params(colors="black")
        ax2.tick_params(colors="black")

        # ── 6. Score Progression ──
        ax = axes[1, 2]
        best_scores, avg_scores = [], []
        for gn in gen_nums:
            gdf = gens[gn]
            if "Docking_Score" in gdf.columns and not gdf.empty:
                best_scores.append(gdf["Docking_Score"].min())
                avg_scores.append(gdf["Docking_Score"].mean())
            else:
                best_scores.append(np.nan)
                avg_scores.append(np.nan)
        ax.plot(gen_nums, best_scores, "o-", color=DARK_COLORS[8], linewidth=2, markersize=6, label="Best Score")
        ax.plot(gen_nums, avg_scores, "s--", color=DARK_COLORS[9], linewidth=2, markersize=6, label="Avg Score")
        ax.set_xlabel("Generation", color="black")
        ax.set_ylabel("Vina Score (kcal/mol)", color="black")
        ax.set_title("6. Score Progression", color="black", fontweight="bold")
        ax.legend(facecolor="white", edgecolor="black")
        ax.tick_params(colors="black")

        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(os.path.join(run_dir, "trial_analysis.png"), dpi=150, bbox_inches="tight",
                    facecolor="white", edgecolor="none")
        display(fig)
        plt.close(fig)

        # ── Summary Table ──
        display(HTML("<hr><h4>Summary Statistics</h4>"))
        summary = []
        for gn in gen_nums:
            gdf = gens[gn]
            row = {"Generation": gn, "Molecules": len(gdf)}
            if "Docking_Score" in gdf.columns and not gdf.empty:
                row["Best_Score"] = f"{gdf['Docking_Score'].min():.2f}"
                row["Avg_Score"] = f"{gdf['Docking_Score'].mean():.2f}"
            if "MW" in gdf.columns:
                row["Avg_MW"] = f"{gdf['MW'].mean():.1f}"
                row[f"Frags<={frag_mw_cap}Da"] = int((gdf["MW"] <= frag_mw_cap).sum())
            if "LogP" in gdf.columns:
                top = gdf.nsmallest(min(top_n, len(gdf)), "Docking_Score") if "Docking_Score" in gdf.columns else gdf.head(top_n)
                row["Top_LogP"] = f"{top['LogP'].mean():.2f}"
                row["Top_TPSA"] = f"{top['TPSA'].mean():.1f}" if "TPSA" in top.columns else "N/A"
            summary.append(row)
        display(pd.DataFrame(summary))
        display(HTML(f"<p><i>Analysis saved to: {os.path.join(run_dir, 'trial_analysis.png')}</i></p>"))

analysis_btn.on_click(run_analysis)
_analysis_refresh.on_click(_refresh_analysis_runs)

display(widgets.VBox([
    widgets.HTML("<h3>Advanced Trial Analysis</h3>"),
    widgets.HTML("<p>Analyzes hybridization success, Pareto frontier, reaction frequency, fragment survival, and physicochemical drift.</p>"),
    widgets.HBox([w_analysis_run, _analysis_refresh]),
    widgets.HBox([w_analysis_topn, w_analysis_mw_threshold]),
    analysis_btn,
    analysis_output
]))


---

## 8. Quick Test Run (Local Validation)

Run a minimal AutoGrow4 test (1 generation, small population) to verify the installation works before submitting a full HPC job.

In [ ]:
import subprocess
import json
import os
from IPython.display import display, HTML
import ipywidgets as widgets

test_output = widgets.Output()
test_btn = widgets.Button(description="Run Quick Test", button_style="warning", layout=widgets.Layout(width="200px"))

def on_test(btn):
    test_output.clear_output()
    with test_output:
        print("Starting quick validation test...")
        print("(1 generation, 3 mutants, 3 crossovers — should complete in a few minutes)\n")

        test_config = {
            "filename_of_receptor": os.path.join(RECEPTOR_DIR, "2ZSH.pdbqt"),
            "center_x": 51.0,
            "center_y": 59.5,
            "center_z": 37.4,
            "size_x": 25.0,
            "size_y": 16.0,
            "size_z": 25.0,
            "source_compound_file": os.path.join(AUTOGROW_DIR, "source_compounds", "naphthalene_smiles.smi"),
            "root_output_folder": os.path.join(PROJECT_DIR, "test_output"),
            "number_of_mutants_first_generation": 3,
            "number_of_crossovers_first_generation": 3,
            "number_of_mutants": 3,
            "number_of_crossovers": 3,
            "number_elitism_advance_from_previous_gen": 3,
            "top_mols_to_seed_next_generation": 3,
            "diversity_mols_to_seed_first_generation": 1,
            "diversity_seed_depreciation_per_gen": 0,
            "num_generations": 1,
            "number_of_processors": 1,
            "multithread_mode": "serial",
            "selector_choice": "Rank_Selector",
            "max_variants_per_compound": 1,
            "filter_source_compounds": False,
            "use_docked_source_compounds": False,
            "LipinskiLenientFilter": True,
            "docking_exhaustiveness": 1,
            "start_a_new_run": True,
            "scoring_choice": "VINA",
            "dock_choice": "QuickVina2Docking",
            "gypsum_timeout_limit": 1,
            "rxn_library": "all_rxns",
        }

        # Add conversion tool
        mgl_path = os.path.join(MGLTOOLS_DIR, "bin", "pythonsh")
        obabel_path = os.popen("which obabel 2>/dev/null").read().strip()

        if os.path.exists(mgl_path):
            test_config["mgltools_directory"] = MGLTOOLS_DIR
            test_config["conversion_choice"] = "MGLToolsConversion"
        elif obabel_path:
            test_config["obabel_path"] = obabel_path
            test_config["conversion_choice"] = "ObabelConversion"
        else:
            print("WARNING: Neither MGLTools nor obabel found!")
            print("Install one of them before running.")
            return

        # Save test config
        test_json = os.path.join(PROJECT_DIR, "test_config.json")
        os.makedirs(test_config["root_output_folder"], exist_ok=True)
        with open(test_json, "w") as f:
            json.dump(test_config, f, indent=2)

        print(f"Config saved to: {test_json}")
        print(f"Output will go to: {test_config['root_output_folder']}")
        print("\nRunning AutoGrow4...\n" + "=" * 60)

        try:
            proc = subprocess.run(
                ["python", "run_autogrow.py", "-j", test_json],
                cwd=AUTOGROW_DIR,
                capture_output=True, text=True,
                timeout=600  # 10 minute timeout
            )

            # Show output
            if proc.stdout:
                for line in proc.stdout.split("\n")[-30:]:
                    print(line)

            if proc.returncode == 0:
                display(HTML("<p style='color:green; font-size:16px'><b>Test completed successfully!</b></p>"))

                # Show results
                import glob
                results = glob.glob(os.path.join(test_config["root_output_folder"], "**", "*ranked*.smi"), recursive=True)
                if results:
                    print(f"\nResult files generated: {len(results)}")
                    for r in results:
                        print(f"  - {r}")
                        with open(r) as rf:
                            lines = rf.readlines()
                            print(f"    ({len(lines)} molecules ranked)")
            else:
                display(HTML("<p style='color:red'><b>Test failed.</b></p>"))
                if proc.stderr:
                    print("\nSTDERR:")
                    for line in proc.stderr.split("\n")[-20:]:
                        print(line)

        except subprocess.TimeoutExpired:
            display(HTML("<span style='color:orange'><b>Test timed out (10 min). This may be normal for slow systems.</b></span>"))
        except Exception as e:
            display(HTML(f"<span style='color:red'>Error: {e}</span>"))

test_btn.on_click(on_test)

display(test_btn)
display(test_output)

---

## 9. Save Results to Project Directory

Copy the final results (ranked molecules, plots, configs) to a clean project output folder.

In [ ]:
import shutil
import glob
import os
from datetime import datetime

results_dir = os.path.join(PROJECT_DIR, f"results_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
os.makedirs(results_dir, exist_ok=True)

output_dir = w_output_folder.value

# Copy ranked .smi files
smi_files = glob.glob(os.path.join(output_dir, "**", "*ranked*.smi"), recursive=True)
for f in smi_files:
    shutil.copy2(f, results_dir)

# Copy any plots
plot_files = glob.glob(os.path.join(output_dir, "**", "*.png"), recursive=True)
for f in plot_files:
    shutil.copy2(f, results_dir)

# Copy config and slurm script
for fname in ["autogrow_config.json", "autogrow_slurm.sh"]:
    src = os.path.join(PROJECT_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, results_dir)

# Copy log files
log_files = glob.glob(os.path.join(PROJECT_DIR, f"{w_slurm_jobname.value}.*.out"))
for f in log_files:
    shutil.copy2(f, results_dir)

print(f"Results saved to: {results_dir}")
print(f"\nContents:")
for f in sorted(os.listdir(results_dir)):
    size = os.path.getsize(os.path.join(results_dir, f))
    print(f"  {f} ({size:,} bytes)")

print(f"\nDone! Your AutoGrow4 run results are saved in:\n  {results_dir}")